# KelpAdam – UTS Pembelajaran Mesin
## Data Preparation & EDA: Preferensi Gaya Belajar (Gasal 2026/2027)

Notebook ini mengolah dataset survei gaya belajar dari dosen sampai siap dianalisis dan siap untuk tahap modelling.

| Bagian | Isi | Status |
|---|---|---|
| 0 | Setup & aturan privasi | ✅ |
| 1 | Data understanding (memahami isi data, mendeteksi masalah) | ✅ |
| 2 | Data cleaning + cleaning log | ⏳ berikutnya |
| 3 | Scoring sesuai teori 4 framework | ⏳ |
| 4 | Data preparation & data dictionary | ⏳ |
| 5 | EDA | ⏳ |
| 6 | Ringkasan, keterbatasan, ekspor output | ⏳ |

**Cara menjalankan:** buka notebook dari folder `notebook/`, lalu pilih *Kernel → Restart & Run All*. Semua path bersifat relatif terhadap folder proyek `UTS/`.

---
# Bagian 0 – Setup

**Aturan yang dipegang di seluruh notebook** (dari spesifikasi proyek):
1. File Excel asli **tidak pernah diubah**. Semua perubahan dilakukan pada salinan di memori lewat kode.
2. Dataset tidak berisi identitas langsung (nama, NPM, email). Kolom `ID` hanya nomor urut.
3. Setiap penghapusan atau perubahan data dicatat di **cleaning log**, beserta jumlah sebelum/sesudah dan alasannya.
4. Outlier **tidak** dihapus otomatis.

In [1]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

# --- Path relatif: notebook ada di UTS/notebook/, data di UTS/dataset/ ---
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
RAW_FILE    = PROJECT_DIR / "dataset" / "raw" / "Survei_Gaya_Belajar_IPS_IPK_NO_PII.xlsx"
REPORT_DIR  = PROJECT_DIR / "report"
PREFIX      = "KelpAdam"

print("Folder proyek :", PROJECT_DIR)
print("File data     :", RAW_FILE.name, "| ada?", RAW_FILE.exists())

Folder proyek : d:\Kuliah\Semester 5\Machine Learning\UTS
File data     : Survei_Gaya_Belajar_IPS_IPK_NO_PII.xlsx | ada? True


---
# Bagian 1 – Data Understanding

Tujuan bagian ini: **mengenal data sebelum mengubah apa pun**. Pertanyaan yang dijawab:
- Seberapa besar datanya dan dari mana asalnya?
- Kolom apa saja yang ada, dan masuk kelompok apa?
- Apa tipe datanya, berapa nilai uniknya, dan berapa rentangnya?
- Di mana ada data kosong atau format yang aneh?

## 1.1 Membaca data & bukti asal-usul (provenance)
Hash MD5 adalah "sidik jari" file. Kalau file Excel berubah sedikit saja, hash-nya ikut berubah. Ini bukti bahwa kita bekerja dengan file asli dari dosen.

In [2]:
raw = pd.read_excel(RAW_FILE)
md5 = hashlib.md5(RAW_FILE.read_bytes()).hexdigest()

print(f"Ukuran data : {raw.shape[0]} baris x {raw.shape[1]} kolom")
print(f"Jumlah sheet: {len(pd.ExcelFile(RAW_FILE).sheet_names)}")
print(f"MD5         : {md5}")

Ukuran data : 194 baris x 116 kolom
Jumlah sheet: 1
MD5         : 390ab027d2a950f63a72684d795caa72


## 1.2 Kelompok kolom
Header kolom berupa teks pertanyaan yang panjang, jadi kolom dikelompokkan berdasarkan **posisinya**. Pengelompokan ini disusun dari urutan kolom dan isi pernyataan (lihat dokumen *Aturan Scoring*, Lampiran 2).

In [3]:
cols = list(raw.columns)
assert len(cols) == 116, "Struktur kolom berbeda dari yang diharapkan"

# (nama kelompok, framework, posisi awal, posisi akhir, kode singkat)
GROUPS = [
    ("ID",                  "Identifier",       0,   0,   "ID"),
    ("VARK 16 pertanyaan",  "VARK",             1,   16,  "VARK_Q"),
    ("Activist",            "Honey-Mumford",    17,  24,  "HM_ACT"),
    ("Reflector",           "Honey-Mumford",    25,  32,  "HM_REF"),
    ("Theorist",            "Honey-Mumford",    33,  40,  "HM_THE"),
    ("Pragmatist",          "Honey-Mumford",    41,  48,  "HM_PRA"),
    ("Active",              "Felder-Silverman", 49,  52,  "FS_ACT"),
    ("Reflective",          "Felder-Silverman", 53,  56,  "FS_REF"),
    ("Sensing",             "Felder-Silverman", 57,  60,  "FS_SEN"),
    ("Intuitive",           "Felder-Silverman", 61,  64,  "FS_INT"),
    ("Visual",              "Felder-Silverman", 65,  68,  "FS_VIS"),
    ("Verbal",              "Felder-Silverman", 69,  72,  "FS_VRB"),
    ("Sequential",          "Felder-Silverman", 73,  76,  "FS_SEQ"),
    ("Global",              "Felder-Silverman", 77,  80,  "FS_GLO"),
    ("CE",                  "Kolb",             81,  86,  "KOLB_CE"),
    ("RO",                  "Kolb",             87,  92,  "KOLB_RO"),
    ("AC",                  "Kolb",             93,  98,  "KOLB_AC"),
    ("AE",                  "Kolb",             99,  104, "KOLB_AE"),
    ("IPS per semester",    "Nilai akademik",   105, 114, "IPS"),
    ("IPK",                 "Nilai akademik",   115, 115, "IPK"),
]

ringkas = pd.DataFrame(
    [(fw, nama, f"{a}–{b}", b - a + 1, cols[a][:70]) for nama, fw, a, b, _ in GROUPS],
    columns=["Framework", "Kelompok", "Posisi kolom", "Jumlah kolom", "Contoh header (dipotong)"],
)
display(ringkas)

VARK_COLS   = cols[1:17]
LIKERT_COLS = cols[17:105]
IPS_COLS    = cols[105:115]
IPK_COL     = cols[115]
SURVEY_COLS = VARK_COLS + LIKERT_COLS

# Kode pendek untuk setiap kolom (dipakai untuk tampilan, lalu untuk rename di C1)
nama_baru = {}
for nama, fw, a, b, kode in GROUPS:
    for k, pos in enumerate(range(a, b + 1), 1):
        asli = cols[pos]
        if kode in ("ID", "IPK"):
            baru = kode
        elif kode == "IPS":
            tahun, smt = asli.split()[1], asli.split()[-1]      # "IPS 2025 Semester 2" -> IPS_2025_S2
            baru = f"IPS_{tahun}_S{smt}"
        elif kode == "VARK_Q":
            baru = f"VARK_Q{k:02d}"
        else:
            baru = f"{kode}_{k:02d}"
        nama_baru[asli] = baru

,Framework,Kelompok,Posisi kolom,Jumlah kolom,Contoh header (dipotong)
0,Identifier,ID,0–0,1,ID
1,VARK,VARK 16 pertanyaan,1–16,16,"Ketika mempelajari topik baru yang cukup kompleks, saya paling terbant"
2,Honey-Mumford,Activist,17–24,8,Saya menikmati aktivitas belajar baru meskipun belum sepenuhnya tahu b
3,Honey-Mumford,Reflector,25–32,8,Saya suka mengamati terlebih dahulu sebelum mengambil kesimpulan.
4,Honey-Mumford,Theorist,33–40,8,"Saya merasa nyaman jika suatu materi memiliki konsep, model, atau kera"
5,Honey-Mumford,Pragmatist,41–48,8,Saya tertarik pada ide yang dapat segera diterapkan untuk memecahkan m
6,Felder-Silverman,Active,49–52,4,Saya memahami materi lebih cepat ketika dapat mendiskusikannya atau me
7,Felder-Silverman,Reflective,53–56,4,Saya lebih suka memikirkan materi secara mendalam sebelum membicarakan
8,Felder-Silverman,Sensing,57–60,4,"Saya lebih menyukai fakta, data, contoh nyata, dan prosedur yang jelas"
9,Felder-Silverman,Intuitive,61–64,4,"Saya tertarik pada kemungkinan, pola, konsep baru, atau pendekatan yan"


## 1.3 Daftar lengkap nama kolom & tipe data
Seluruh 116 kolom beserta kode pendeknya, tipe data, jumlah sel terisi/kosong, dan jumlah nilai unik.

In [4]:
daftar_kolom = pd.DataFrame({
    "Posisi": range(len(cols)),
    "Kode pendek": [nama_baru[c] for c in cols],
    "Nama kolom asli": cols,
    "Tipe data": [str(raw[c].dtype) for c in cols],
    "Terisi": raw.notna().sum().values,
    "Kosong": raw.isna().sum().values,
    "Nilai unik": raw.nunique().values,
})
with pd.option_context("display.max_rows", None, "display.max_colwidth", 110):
    display(daftar_kolom)

,Posisi,Kode pendek,Nama kolom asli,Tipe data,Terisi,Kosong,Nilai unik
0,0,ID,ID,int64,194,0,194
1,1,VARK_Q01,"Ketika mempelajari topik baru yang cukup kompleks, saya paling terbantu jika...",str,192,2,41
2,2,VARK_Q02,"Saat harus memahami prosedur penggunaan suatu perangkat atau aplikasi, saya cenderung...",str,192,2,35
3,3,VARK_Q03,"Untuk mengingat materi menjelang ujian, saya biasanya lebih suka...",str,192,2,30
4,4,VARK_Q04,"Ketika dosen menjelaskan konsep abstrak, saya lebih mudah memahami bila...",str,192,2,33
5,5,VARK_Q05,"Saat menerima tugas baru, saya biasanya ingin terlebih dahulu...",str,192,2,35
6,6,VARK_Q06,"Jika saya tersesat di tempat yang belum dikenal, saya lebih memilih...",str,192,2,39
7,7,VARK_Q07,"Saat mempelajari data atau hasil penelitian, saya paling terbantu oleh...",str,192,2,39
8,8,VARK_Q08,"Jika harus menjelaskan ide kepada teman, saya cenderung...",str,192,2,32
9,9,VARK_Q09,"Dalam kelas daring, saya paling tertarik ketika materi menyediakan...",str,192,2,31


**Cara membaca tipe data:**
- `int64` = bilangan bulat (ID).
- `str`/`object` = teks (jawaban VARK).
- `float64` = angka desimal. Pernyataan skala 1–5 tersimpan sebagai desimal karena ada sel kosong (pandas mengubah kolom berisi `NaN` menjadi float), padahal isinya bilangan bulat. Tipe ini diperbaiki di cleaning (C3).

## 1.4 Contoh baris
Lima baris pertama untuk beberapa kolom (ID, 2 pertanyaan VARK, 2 pernyataan skala 1–5, 2 kolom IPS, IPK). Header ditampilkan memakai kode pendek supaya terbaca; teks lengkapnya ada di tabel 1.3.

In [5]:
contoh_cols = [cols[0], cols[1], cols[2], cols[17], cols[18], cols[108], cols[113], cols[115]]
with pd.option_context("display.max_colwidth", 60):
    display(raw[contoh_cols].rename(columns=nama_baru).head(5))

,ID,VARK_Q01,VARK_Q02,HM_ACT_01,HM_ACT_02,IPS_2023_S2,IPS_2025_S3,IPK
0,1,"melihat diagram, bagan, peta konsep, atau ilustrasi hubu...",meminta seseorang menjelaskan langkah-langkahnya secara ...,3.0,4.0,NaN,NaN,3.78
1,2,"melihat diagram, bagan, peta konsep, atau ilustrasi hubu...",mengikuti gambar langkah demi langkah atau tangkapan lay...,4.0,3.0,NaN,NaN,3.83
2,3,"melihat diagram, bagan, peta konsep, atau ilustrasi hubu...",langsung mencoba sambil memperbaiki kesalahan yang muncu...,5.0,4.0,NaN,NaN,3.77
3,4,"mencoba contoh, simulasi, praktik, atau melihat penerapa...",langsung mencoba sambil memperbaiki kesalahan yang muncul.;,5.0,5.0,NaN,NaN,1.97
4,5,mendengar penjelasan dan dapat berdiskusi atau bertanya ...,mengikuti gambar langkah demi langkah atau tangkapan lay...,4.0,3.0,NaN,NaN,3.54


## 1.5 Data kosong (missing values)
Pertama dilihat per kelompok, lalu per kolom, lalu pola kekosongannya untuk mencari tahu **mengapa** datanya kosong.

In [6]:
def info_kelompok(nama, kolom):
    sub = raw[kolom]
    return {
        "Kelompok": nama,
        "Jumlah kolom": len(kolom),
        "Tipe data": ", ".join(sorted({str(t) for t in sub.dtypes})),
        "Sel kosong": int(sub.isna().sum().sum()),
        "Kolom yg punya sel kosong": int(sub.isna().any().sum()),
        "Nilai unik (min–maks per kolom)": f"{sub.nunique().min()}–{sub.nunique().max()}",
    }

tabel_info = pd.DataFrame([
    info_kelompok("ID", [cols[0]]),
    info_kelompok("VARK (16)", VARK_COLS),
    info_kelompok("Likert 1–5 (88)", LIKERT_COLS),
    info_kelompok("IPS (10)", IPS_COLS),
    info_kelompok("IPK", [IPK_COL]),
])
display(tabel_info)

print("ID unik?", raw[cols[0]].is_unique, "| rentang ID:", raw[cols[0]].min(), "–", raw[cols[0]].max())

,Kelompok,Jumlah kolom,Tipe data,Sel kosong,Kolom yg punya sel kosong,Nilai unik (min–maks per kolom)
0,ID,1,int64,0,0,194–194
1,VARK (16),16,str,32,16,30–41
2,Likert 1–5 (88),88,float64,176,88,4–5
3,IPS (10),10,float64,1118,10,1–139
4,IPK,1,float64,17,1,98–98


ID unik? True | rentang ID: 1 – 194


### a) Kolom survei: dari mana sel kosongnya?

In [7]:
kosong_survei = raw[SURVEY_COLS].isna().sum()
print("Jumlah sel kosong per kolom survei:", sorted(kosong_survei.unique().tolist()),
      f"(berlaku untuk {(kosong_survei > 0).sum()} dari {len(SURVEY_COLS)} kolom)")

baris_kosong_total = raw[SURVEY_COLS].isna().all(axis=1)
print("Baris yang SELURUH jawaban surveinya kosong:", raw.loc[baris_kosong_total, cols[0]].tolist())
print("Baris yang kosong SEBAGIAN:",
      int((raw[SURVEY_COLS].isna().any(axis=1) & ~baris_kosong_total).sum()))
display(raw.loc[baris_kosong_total, [cols[0]] + IPS_COLS + [IPK_COL]].rename(columns=nama_baru))

Jumlah sel kosong per kolom survei: [2] (berlaku untuk 104 dari 104 kolom)
Baris yang SELURUH jawaban surveinya kosong: [11, 133]
Baris yang kosong SEBAGIAN: 0


,ID,IPS_2022_S1,IPS_2022_S2,IPS_2023_S1,IPS_2023_S2,IPS_2024_S1,IPS_2024_S2,IPS_2025_S1,IPS_2025_S2,IPS_2025_S3,IPS_2026_S1,IPK
10,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
132,133,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Temuan:** setiap kolom survei kosong tepat 2 sel, dan semuanya berasal dari **2 baris yang kosong total** (ID 11 dan 133). IPS dan IPK keduanya juga kosong. Tidak ada responden yang mengisi sebagian lalu berhenti. Kedua baris ini dibuang di cleaning (C2).

### b) IPS dan IPK: kosong per kolom

In [8]:
kosong_akad = pd.DataFrame({
    "Terisi": raw[IPS_COLS + [IPK_COL]].notna().sum(),
    "Kosong": raw[IPS_COLS + [IPK_COL]].isna().sum(),
})
kosong_akad["% kosong"] = (kosong_akad["Kosong"] / len(raw) * 100).round(1)
display(kosong_akad.rename(index=nama_baru))

,Terisi,Kosong,% kosong
IPS_2022_S1,1,193,99.5
IPS_2022_S2,1,193,99.5
IPS_2023_S1,18,176,90.7
IPS_2023_S2,18,176,90.7
IPS_2024_S1,125,69,35.6
IPS_2024_S2,125,69,35.6
IPS_2025_S1,173,21,10.8
IPS_2025_S2,172,22,11.3
IPS_2025_S3,16,178,91.8
IPS_2026_S1,173,21,10.8


### c) Pola kekosongan IPS: missing struktural
Kalau kosongnya IPS disebabkan angkatan, polanya pasti teratur: mahasiswa angkatan 2024 punya IPS mulai 2024, angkatan 2025 mulai 2025, dan seterusnya. Tabel di bawah menghitung **tahun IPS pertama** setiap responden (IPS 2026 S1 tidak dihitung karena semester masih berjalan).

In [9]:
ips_selesai_raw = IPS_COLS[:-1]                                   # tanpa IPS 2026 Semester 1
punya_ips = raw[ips_selesai_raw].notna()
tahun_pertama = punya_ips.idxmax(axis=1).str.split().str[1].where(punya_ips.any(axis=1))
pengganti = pd.Series(np.where(raw[IPS_COLS[-1]].notna(), "2026 (hanya IPS semester berjalan)",
                               "tidak ada IPS sama sekali"), index=raw.index)
tahun_pertama = tahun_pertama.fillna(pengganti)
tahun_pertama[baris_kosong_total] = "baris kosong total"

def ada_celah(baris):
    idx = np.where(baris)[0]
    return len(idx) > 0 and (idx[-1] - idx[0] + 1) != len(idx)

pola = (pd.DataFrame({"Tahun IPS pertama": tahun_pertama, "IPK kosong": raw[IPK_COL].isna()})
        .groupby("Tahun IPS pertama").agg(Responden=("IPK kosong", "size"), IPK_kosong=("IPK kosong", "sum")))
display(pola)
print("Responden yang IPS-nya bolong di tengah (cuti?):",
      raw.loc[punya_ips.apply(lambda r: ada_celah(r.to_numpy()), axis=1), cols[0]].tolist())

,Responden,IPK_kosong
Tahun IPS pertama,,
2022,1,0
2023,17,0
2024,107,0
2025,48,0
2026 (hanya IPS semester berjalan),11,7
baris kosong total,2,2
tidak ada IPS sama sekali,8,8


Responden yang IPS-nya bolong di tengah (cuti?): []


**Temuan:** kekosongan IPS mengikuti pola angkatan, jadi termasuk **missing struktural**, bukan responden lupa mengisi. Konsekuensinya:
- IPS **tidak boleh diimputasi** (diisi tebakan), karena nilainya memang belum ada.
- Tahun IPS pertama bisa dipakai untuk **memperkirakan angkatan** (derived variable).
- Responden tanpa IPS sama sekali juga tidak punya IPK, sehingga analisis yang memakai IPK hanya menggunakan responden yang IPK-nya ada.

### d) Kolom yang kosong seluruhnya atau isinya hanya satu nilai
Kolom yang **seluruhnya kosong** tidak berguna sama sekali. Kolom **konstan** (hanya satu nilai) juga tidak memberi informasi, karena semua orang nilainya sama.

In [10]:
kolom_kosong_total = [nama_baru[c] for c in cols if raw[c].isna().all()]
kolom_konstan = pd.DataFrame(
    [(nama_baru[c], raw[c].dropna().unique()[0], int(raw[c].notna().sum())) for c in cols if raw[c].nunique() == 1],
    columns=["Kolom", "Satu-satunya nilai", "Jumlah terisi"],
)
print("Kolom yang seluruhnya kosong:", kolom_kosong_total if kolom_kosong_total else "tidak ada")
print("Kolom yang hanya punya 1 nilai unik:")
display(kolom_konstan)

Kolom yang seluruhnya kosong: tidak ada
Kolom yang hanya punya 1 nilai unik:


,Kolom,Satu-satunya nilai,Jumlah terisi
0,IPS_2022_S1,3.026315,1
1,IPS_2022_S2,3.133333,1
2,IPS_2026_S1,0.000000,173


**Temuan:**
- **Tidak ada kolom yang kosong seluruhnya.**
- **IPS 2026 Semester 1** hanya berisi angka 0 untuk semua yang terisi. Ini bukan IPS nol, tetapi "nilai belum ada" karena semester masih berjalan, sehingga kolom ini tidak informatif.
- **IPS 2022 Semester 1 & 2** hanya terisi oleh 1 orang (satu-satunya angkatan 2022), jadi wajar kalau nilainya hanya satu. Ini bukan masalah format.

## 1.6 Deteksi duplikat
Di tahap ini duplikat hanya **dideteksi**, belum dihapus. Ada empat tingkat pengecekan:
1. **ID ganda:** apakah ada nomor ID yang muncul dua kali?
2. **Duplikat persis:** semua kolom sama (selain ID).
3. **Jawaban survei identik:** ke-104 jawaban survei sama persis.
4. **Sangat mirip:** paling banyak 5 dari 88 jawaban skala 1–5 yang berbeda. Kalau ditemukan, IPS/IPK-nya dibandingkan. Nilai akademik berasal dari data kampus, jadi kalau IPS/IPK berbeda, hampir pasti orangnya berbeda.

In [11]:
id_col = cols[0]
print("1) ID ganda:", int(raw[id_col].duplicated().sum()))

dup_persis = raw.drop(columns=id_col).duplicated(keep=False)
print("2) Baris yang duplikat persis (selain ID):", raw.loc[dup_persis, id_col].tolist())

terisi = raw.loc[~baris_kosong_total]                     # abaikan 2 baris kosong total
dup_survei = terisi[SURVEY_COLS].duplicated(keep=False)
print("3) Jawaban survei identik (di antara baris yang terisi):", terisi.loc[dup_survei, id_col].tolist())

Lk = terisi[LIKERT_COLS].to_numpy(dtype=float)
id_t = terisi[id_col].to_numpy()
mirip = []
for i in range(len(Lk)):
    beda = (Lk[i] != Lk[i + 1:]).sum(axis=1)
    for j in np.where(beda <= 5)[0]:
        a, b = terisi.iloc[i], terisi.iloc[i + 1 + j]
        mirip.append({
            "ID a": id_t[i], "ID b": id_t[i + 1 + j],
            "Likert beda (dari 88)": int(beda[j]),
            "VARK beda (dari 16)": int((a[VARK_COLS] != b[VARK_COLS]).sum()),
            "IPS/IPK sama?": bool(pd.concat([a[IPS_COLS + [IPK_COL]], b[IPS_COLS + [IPK_COL]]], axis=1)
                                  .nunique(axis=1, dropna=False).max() == 1),
        })
tabel_mirip = pd.DataFrame(mirip)
print(f"4) Pasangan sangat mirip: {len(tabel_mirip)} pasang; yang IPS/IPK-nya juga sama: {int(tabel_mirip['IPS/IPK sama?'].sum())}")
display(tabel_mirip.sort_values(["VARK beda (dari 16)", "Likert beda (dari 88)"]).head(10))

1) ID ganda: 0
2) Baris yang duplikat persis (selain ID): [11, 133]
3) Jawaban survei identik (di antara baris yang terisi): []
4) Pasangan sangat mirip: 78 pasang; yang IPS/IPK-nya juga sama: 0


,ID a,ID b,Likert beda (dari 88),VARK beda (dari 16),IPS/IPK sama?
49,78,188,4,1,False
56,125,132,0,3,False
29,55,132,0,10,False
28,55,125,0,11,False
65,136,159,0,11,False
10,38,78,1,11,False
13,38,188,3,11,False
53,115,159,0,12,False
43,68,136,0,14,False
75,167,174,0,14,False


**Mengapa banyak pasangan yang "sangat mirip"?** Ternyata sebagian responden menjawab **angka yang sama untuk seluruh 88 pernyataan** (*straightlining*), atau hampir seluruhnya (84–87 dari 88). Kalau dua orang sama-sama menjawab "4" untuk hampir semua pernyataan, jawaban skala mereka otomatis sangat mirip walaupun orangnya berbeda.

In [12]:
jawaban_terbanyak = terisi[LIKERT_COLS].apply(lambda r: r.value_counts().iloc[0], axis=1)
seragam = terisi[LIKERT_COLS].nunique(axis=1).eq(1)            # 88 dari 88 sama
hampir  = ~seragam & (jawaban_terbanyak >= 84)                  # 84-87 dari 88 sama
ids_seragam = set(terisi.loc[seragam, id_col])
ids_hampir  = set(terisi.loc[hampir, id_col])
print("Straightliner (88 jawaban identik):", len(ids_seragam), "orang ->", sorted(ids_seragam))
print("  angka yang dipilih:", {int(k): int(v) for k, v in terisi.loc[seragam, LIKERT_COLS[0]].value_counts().sort_index().items()})
print("Hampir straightline (84-87 identik):", len(ids_hampir), "orang ->", sorted(ids_hampir))

def jenis_pasangan(r):
    grup = ids_seragam | ids_hampir
    if r["ID a"] in ids_seragam and r["ID b"] in ids_seragam:
        return "keduanya straightliner"
    if r["ID a"] in grup and r["ID b"] in grup:
        return "keduanya (hampir) seragam"
    if r["ID a"] in grup or r["ID b"] in grup:
        return "salah satu (hampir) seragam"
    return "keduanya menjawab bervariasi"

tabel_mirip["Jenis pasangan"] = tabel_mirip.apply(jenis_pasangan, axis=1)
rekap_pasangan = tabel_mirip.groupby("Jenis pasangan").agg(
    Pasangan=("ID a", "size"),
    VARK_beda_min=("VARK beda (dari 16)", "min"),
    IPS_IPK_sama=("IPS/IPK sama?", "sum"),
)
display(rekap_pasangan)

Straightliner (88 jawaban identik): 15 orang -> [9, 38, 55, 57, 64, 68, 115, 125, 132, 136, 159, 167, 171, 174, 194]
  angka yang dipilih: {3: 5, 4: 5, 5: 5}
Hampir straightline (84-87 identik): 8 orang -> [42, 44, 78, 126, 142, 166, 187, 188]


,Pasangan,VARK_beda_min,IPS_IPK_sama
Jenis pasangan,,,
keduanya (hampir) seragam,48,1,0
keduanya straightliner,30,3,0


**Temuan deteksi duplikat:**
- **Tidak ada ID ganda.**
- Duplikat persis hanya muncul pada **ID 11 dan 133**. Keduanya "kembar" karena sama-sama kosong total, bukan karena orang yang sama mengisi dua kali. Keduanya akan dibuang di cleaning sebagai respons kosong.
- **Tidak ada jawaban survei yang identik** di antara responden yang mengisi.
- Semua pasangan yang "sangat mirip" melibatkan responden yang jawabannya **seragam atau hampir seragam**. Tidak ada pasangan mirip di antara responden yang menjawab bervariasi.
- **IPS/IPK semua pasangan berbeda**, dan jawaban VARK-nya pun umumnya berbeda jauh. Pasangan paling mirip, ID 78 & 188 (Likert beda 4, VARK beda 1), tetap punya IPS/IPK berbeda. Kesimpulannya, **tidak ada duplikat responden**.
- Temuan **straightlining** (15 orang) dan **hampir straightline** (8 orang) adalah masalah kualitas jawaban, bukan duplikat. Penanganannya diputuskan di cleaning.

## 1.7 VARK: format jawaban & kategori yang muncul
Setiap jawaban VARK berupa teks opsi yang dipisah tanda `;`, karena responden boleh memilih lebih dari satu. Yang dicek: berapa opsi unik per pertanyaan, berapa opsi yang dipilih orang per pertanyaan, dan opsi apa saja yang muncul.

In [13]:
def pecah(sel):
    """'opsi a;opsi b;' -> ['opsi a', 'opsi b']"""
    return [p.strip() for p in str(sel).split(";") if p.strip()]

rekap_vark = []
for i, c in enumerate(VARK_COLS, 1):
    isi = raw[c].dropna()
    opsi_unik = {o for sel in isi for o in pecah(sel)}
    n_pilih = isi.apply(lambda s: len(pecah(s)))
    rekap_vark.append({
        "Pertanyaan": f"Q{i:02d}",
        "Opsi unik": len(opsi_unik),
        "Min dipilih": n_pilih.min(),
        "Maks dipilih": n_pilih.max(),
        "Rata-rata dipilih": round(n_pilih.mean(), 2),
        "Selalu diakhiri ';'?": bool(isi.str.endswith(";").all()),
    })
display(pd.DataFrame(rekap_vark))

,Pertanyaan,Opsi unik,Min dipilih,Maks dipilih,Rata-rata dipilih,Selalu diakhiri ';'?
0,Q01,4,1,4,2.11,True
1,Q02,4,1,4,1.89,True
2,Q03,4,1,4,1.92,True
3,Q04,4,1,4,1.98,True
4,Q05,4,1,4,1.92,True
5,Q06,4,1,4,2.03,True
6,Q07,4,1,4,2.06,True
7,Q08,4,1,4,1.87,True
8,Q09,4,1,4,1.79,True
9,Q10,4,1,4,1.91,True


**Kategori yang muncul:** semua opsi yang ada di setiap pertanyaan, beserta berapa responden yang memilihnya. Karena boleh memilih lebih dari satu, total persentase per pertanyaan bisa lebih dari 100%.

In [14]:
n_jawab = raw[VARK_COLS[0]].notna().sum()
frek = []
for i, c in enumerate(VARK_COLS, 1):
    hitung = pd.Series([o for s in raw[c].dropna() for o in pecah(s)]).value_counts()
    for opsi, n in hitung.items():
        frek.append({"Pertanyaan": f"Q{i:02d}", "Opsi": opsi.rstrip("."), "Dipilih": int(n),
                     "% responden": round(n / n_jawab * 100, 1)})
frek_vark = pd.DataFrame(frek)
with pd.option_context("display.max_rows", None, "display.max_colwidth", 100):
    display(frek_vark)

,Pertanyaan,Opsi,Dipilih,% responden
0,Q01,"mencoba contoh, simulasi, praktik, atau melihat penerapan nyata",121,63.0
1,Q01,mendengar penjelasan dan dapat berdiskusi atau bertanya secara lisan,113,58.9
2,Q01,"melihat diagram, bagan, peta konsep, atau ilustrasi hubungan antarbagian",92,47.9
3,Q01,"membaca uraian tertulis, definisi, atau catatan yang terstruktur",80,41.7
4,Q02,mengikuti gambar langkah demi langkah atau tangkapan layar,107,55.7
5,Q02,membaca petunjuk tertulis dan daftar langkah,94,49.0
6,Q02,langsung mencoba sambil memperbaiki kesalahan yang muncul,89,46.4
7,Q02,meminta seseorang menjelaskan langkah-langkahnya secara lisan,72,37.5
8,Q03,"berlatih soal, mencoba contoh, atau menghubungkan materi dengan pengalaman",131,68.2
9,Q03,"menulis ulang ringkasan, membuat daftar, atau membaca catatan",128,66.7


**Temuan:** setiap pertanyaan punya **tepat 4 opsi**, masing-masing mewakili satu modalitas V/A/R/K (pemetaannya ditetapkan di C4). Tidak ada isian bebas atau typo, dan orang memilih 1 sampai 4 opsi per pertanyaan. Opsi yang paling sering dipilih di antaranya "menjelaskan secara lisan" (Q12) dan "berlatih soal/mencoba contoh" (Q03).

## 1.8 Pernyataan skala 1–5: rentang, distribusi, dan statistik ringkas

In [15]:
nilai = pd.Series(raw[LIKERT_COLS].values.ravel()).dropna()
print("Nilai minimum:", nilai.min(), "| maksimum:", nilai.max())
print("Ada nilai desimal?", bool((nilai % 1 != 0).any()))

dist = nilai.value_counts().sort_index().rename("jumlah").to_frame()
dist["persen"] = (dist["jumlah"] / dist["jumlah"].sum() * 100).round(1)
display(dist)

Nilai minimum: 1.0 | maksimum: 5.0
Ada nilai desimal? False


,jumlah,persen
1.0,166,1.0
2.0,710,4.2
3.0,4387,26.0
4.0,6862,40.6
5.0,4771,28.2


**Statistik ringkas per pernyataan** (88 item). Kolom `% setuju` = persentase jawaban 4 atau 5.

In [16]:
L = raw[LIKERT_COLS]
stat_likert = pd.DataFrame({
    "Kode": [nama_baru[c] for c in LIKERT_COLS],
    "Pernyataan": [c[:75] for c in LIKERT_COLS],
    "n": L.notna().sum().values,
    "Rata-rata": L.mean().round(2).values,
    "SD": L.std().round(2).values,
    "Median": L.median().values,
    "Min": L.min().values,
    "Maks": L.max().values,
    "Nilai unik": L.nunique().values,
    "% setuju": ((L >= 4).sum() / L.notna().sum() * 100).round(1).values,
})
with pd.option_context("display.max_rows", None, "display.max_colwidth", 80):
    display(stat_likert)

print("Rata-rata item terendah :", stat_likert.nsmallest(1, "Rata-rata")[["Kode", "Rata-rata"]].values.tolist())
print("Rata-rata item tertinggi:", stat_likert.nlargest(1, "Rata-rata")[["Kode", "Rata-rata"]].values.tolist())
print("Rentang % setuju per item:", stat_likert["% setuju"].min(), "–", stat_likert["% setuju"].max())

,Kode,Pernyataan,n,Rata-rata,SD,Median,Min,Maks,Nilai unik,% setuju
0,HM_ACT_01,Saya menikmati aktivitas belajar baru meskipun belum sepenuhnya tahu bagaim,192,3.79,0.88,4.0,1.0,5.0,5,66.1
1,HM_ACT_02,Saya lebih bersemangat ketika dapat langsung terlibat dalam tugas atau tant,192,3.89,0.88,4.0,1.0,5.0,5,65.1
2,HM_ACT_03,Saya suka mencoba berbagai cara tanpa terlalu lama menunggu sebelum memulai,192,3.71,0.93,4.0,1.0,5.0,5,57.3
3,HM_ACT_04,"Diskusi, simulasi, atau aktivitas kelompok membuat saya lebih aktif belajar",192,3.85,0.87,4.0,1.0,5.0,5,67.7
4,HM_ACT_05,Saya tertarik pada pengalaman yang menuntut spontanitas dan keterlibatan la,192,3.94,0.89,4.0,1.0,5.0,5,70.3
5,HM_ACT_06,Saya cenderung menerima tantangan baru sebagai kesempatan untuk belajar.,192,3.83,0.91,4.0,1.0,5.0,5,64.1
6,HM_ACT_07,Saya merasa belajar paling hidup ketika banyak hal terjadi dan saya dapat i,192,3.98,0.88,4.0,1.0,5.0,5,70.3
7,HM_ACT_08,Saya mudah tertarik mencoba metode atau alat baru dalam pembelajaran.,192,3.93,0.84,4.0,1.0,5.0,5,71.4
8,HM_REF_01,Saya suka mengamati terlebih dahulu sebelum mengambil kesimpulan.,192,4.18,0.81,4.0,2.0,5.0,4,79.7
9,HM_REF_02,Saya mempertimbangkan pengalaman dari beberapa sudut pandang sebelum bertin,192,4.09,0.83,4.0,1.0,5.0,5,76.6


Rata-rata item terendah : [['HM_REF_06', 3.26]]
Rata-rata item tertinggi: [['HM_THE_03', 4.24]]
Rentang % setuju per item: 38.5 – 82.3


**Temuan:** semua jawaban bulat 1–5, tanpa nilai aneh. Sekitar **69% jawaban adalah 4 atau 5**, dan rata-rata setiap item berada di kisaran 3,3–4,2. Responden cenderung menjawab "setuju", dan ini penting saat menafsirkan skor nanti. Beberapa item tidak pernah dijawab 1 sehingga hanya punya 4 nilai unik. Itu wajar dan bukan error.

## 1.9 IPS dan IPK

In [17]:
display(raw[IPS_COLS + [IPK_COL]].describe().T.round(2))
print("Nilai IPS 2026 Semester 1 yang muncul:", raw[IPS_COLS[-1]].value_counts(dropna=False).to_dict())

,count,mean,std,min,25%,50%,75%,max
IPS 2022 Semester 1,1.0,3.03,NaN,3.03,3.03,3.03,3.03,3.03
IPS 2022 Semester 2,1.0,3.13,NaN,3.13,3.13,3.13,3.13,3.13
IPS 2023 Semester 1,18.0,3.49,0.38,2.68,3.20,3.62,3.73,3.97
IPS 2023 Semester 2,18.0,3.55,0.33,2.83,3.36,3.56,3.80,4.00
IPS 2024 Semester 1,125.0,3.33,0.37,2.26,3.13,3.34,3.59,4.00
IPS 2024 Semester 2,125.0,3.37,0.50,0.95,3.18,3.44,3.71,4.00
IPS 2025 Semester 1,173.0,3.29,0.56,0.87,3.01,3.40,3.72,4.00
IPS 2025 Semester 2,172.0,3.47,0.46,0.94,3.22,3.59,3.82,4.00
IPS 2025 Semester 3,16.0,3.47,0.43,3.00,3.00,3.50,3.89,4.00
IPS 2026 Semester 1,173.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00


Nilai IPS 2026 Semester 1 yang muncul: {0.0: 173, nan: 21}


**Temuan:**
- Semua nilai berada di rentang 0–4.
- **IPS 2026 Semester 1 bernilai 0 untuk semua orang** karena semester ini masih berjalan. Angka 0 di sini berarti "belum ada nilai", bukan IPS nol.
- Ada IPK = 0 pada mahasiswa baru yang belum punya IPS sama sekali.

Keduanya akan diperbaiki di tahap cleaning.

## 1.10 Ringkasan: variabel dikelompokkan berdasarkan peran
Spesifikasi (Bagian 7.1) meminta variabel dikelompokkan berdasarkan **peran/jenisnya**: metadata non-identifying, item survei, skor numerik, dan label/kategori. Tabel di bawah juga merangkum nilai unik, kategori yang muncul, rentang, dan konsistensi format setiap kelompok.

In [18]:
def rentang(kolom):
    v = raw[kolom].stack()
    return f"{v.min():g} – {v.max():g}"

def desimal_maks(kolom):
    v = raw[kolom].stack()
    return int(v.map(lambda x: len(f"{x}".split(".")[1].rstrip("0")) if "." in f"{x}" else 0).max())

opsi_per_soal = sorted({len({o for s in raw[c].dropna() for o in pecah(s)}) for c in VARK_COLS})
kombinasi = raw[VARK_COLS].nunique()

peran = pd.DataFrame([
    {
        "Peran": "Identifier (bukan PII)",
        "Kolom": "ID", "Jumlah": 1, "Tipe": "integer",
        "Nilai unik / kategori": f"{raw[cols[0]].nunique()} nilai unik (1 per baris)",
        "Rentang": rentang([cols[0]]),
        "Konsistensi format": "Unik, berurutan, tanpa celah; hanya nomor urut",
    },
    {
        "Peran": "Metadata non-identifying (konteks akademik)",
        "Kolom": "IPS per semester, IPK", "Jumlah": len(IPS_COLS) + 1, "Tipe": "numerik kontinu",
        "Nilai unik / kategori": f"IPS {raw[IPS_COLS].nunique().min()}–{raw[IPS_COLS].nunique().max()} per kolom; IPK {raw[IPK_COL].nunique()}",
        "Rentang": f"IPS {rentang(IPS_COLS)}; IPK {rentang([IPK_COL])}",
        "Konsistensi format": f"IPS sampai {desimal_maks(IPS_COLS)} desimal, IPK {desimal_maks([IPK_COL])} desimal; "
                              "IPS 2026 S1 semua 0 (semester berjalan); IPK 0 pada mhs baru",
    },
    {
        "Peran": "Item survei – kategorikal multi-jawab",
        "Kolom": "VARK_Q01–Q16", "Jumlah": len(VARK_COLS), "Tipe": "teks (opsi dipisah ';')",
        "Nilai unik / kategori": f"{opsi_per_soal[0]} opsi per soal (V/A/R/K); {kombinasi.min()}–{kombinasi.max()} kombinasi jawaban per kolom",
        "Rentang": "1–4 opsi dipilih per soal",
        "Konsistensi format": "Semua diakhiri ';', tanpa typo/isian bebas",
    },
    {
        "Peran": "Item survei – ordinal (Likert 1–5)",
        "Kolom": "HM/FS/KOLB (88 pernyataan)", "Jumlah": len(LIKERT_COLS), "Tipe": "ordinal (tersimpan float)",
        "Nilai unik / kategori": f"{raw[LIKERT_COLS].nunique().min()}–{raw[LIKERT_COLS].nunique().max()} nilai per kolom dari {{1,2,3,4,5}}",
        "Rentang": rentang(LIKERT_COLS),
        "Konsistensi format": "Semua bulat, tanpa nilai di luar 1–5",
    },
    {
        "Peran": "Skor numerik",
        "Kolom": "—", "Jumlah": 0, "Tipe": "—",
        "Nilai unik / kategori": "Belum ada di dataset asli",
        "Rentang": "—",
        "Konsistensi format": "Dibuat di Bagian 3 (scoring sesuai teori)",
    },
    {
        "Peran": "Label / kategori gaya belajar",
        "Kolom": "—", "Jumlah": 0, "Tipe": "—",
        "Nilai unik / kategori": "Belum ada di dataset asli",
        "Rentang": "—",
        "Konsistensi format": "Dibuat di Bagian 3 (scoring sesuai teori)",
    },
])
assert peran["Jumlah"].sum() == raw.shape[1]
display(peran)

,Peran,Kolom,Jumlah,Tipe,Nilai unik / kategori,Rentang,Konsistensi format
0,Identifier (bukan PII),ID,1,integer,194 nilai unik (1 per baris),1 – 194,"Unik, berurutan, tanpa celah; hanya nomor urut"
1,Metadata non-identifying (konteks akademik),"IPS per semester, IPK",11,numerik kontinu,IPS 1–139 per kolom; IPK 98,IPS 0 – 4; IPK 0 – 4,"IPS sampai 6 desimal, IPK 2 desimal; IPS 2026 S1 semua 0 (semester berjalan)..."
2,Item survei – kategorikal multi-jawab,VARK_Q01–Q16,16,teks (opsi dipisah ';'),4 opsi per soal (V/A/R/K); 30–41 kombinasi jawaban per kolom,1–4 opsi dipilih per soal,"Semua diakhiri ';', tanpa typo/isian bebas"
3,Item survei – ordinal (Likert 1–5),HM/FS/KOLB (88 pernyataan),88,ordinal (tersimpan float),"4–5 nilai per kolom dari {1,2,3,4,5}",1 – 5,"Semua bulat, tanpa nilai di luar 1–5"
4,Skor numerik,—,0,—,Belum ada di dataset asli,—,Dibuat di Bagian 3 (scoring sesuai teori)
5,Label / kategori gaya belajar,—,0,—,Belum ada di dataset asli,—,Dibuat di Bagian 3 (scoring sesuai teori)


**Kesimpulan data understanding:**
- Dataset berisi **item survei mentah** (16 VARK + 88 Likert) ditambah **metadata akademik** (IPS, IPK) dan **ID**. Totalnya 116 kolom.
- **Belum ada skor numerik maupun label.** Keduanya harus dibuat sendiri dengan aturan teori masing-masing framework (Bagian 3).
- **Tidak ada metadata demografis** seperti prodi, angkatan, atau waktu pengisian. Angkatan hanya bisa diperkirakan dari tahun IPS pertama, dan perkiraan ini akan diperlakukan sebagai *derived variable*.
- Formatnya konsisten. Masalah yang perlu dibersihkan hanya 2 baris kosong total dan angka 0 yang sebenarnya berarti "belum ada nilai" di IPS/IPK.

## 1.11 Ringkasan temuan & apa yang akan dilakukan di cleaning
Bagian ini hanya **mendeteksi**. Semua perbaikan dikerjakan di Bagian 2 (cleaning) dan dicatat di cleaning log.

| # | Temuan data understanding | Bagian | Rencana di cleaning |
|---|---|---|---|
| 1 | Header berupa teks pertanyaan yang panjang | 1.3 | Ganti nama ke kode pendek; teks asli disimpan untuk data dictionary |
| 2 | 2 baris kosong total (ID 11, 133), yang juga terdeteksi sebagai "duplikat" satu sama lain | 1.5, 1.6 | **Hapus** (respons kosong) |
| 3 | Pernyataan skala 1–5 tersimpan sebagai desimal | 1.3, 1.8 | Validasi nilai 1–5, ubah tipe ke bilangan bulat |
| 4 | Jawaban VARK diakhiri `;`, 4 opsi per soal | 1.7 | Validasi opsi terhadap daftar resmi, rapikan format |
| 5 | IPS 2026 S1 = 0 untuk semua (semester berjalan) | 1.5d, 1.9 | Ubah 0 → kosong (`NaN`) |
| 6 | IPK = 0 pada mahasiswa baru yang belum punya IPS | 1.9 | Ubah 0 → kosong agar konsisten |
| 7 | IPS banyak kosong karena angkatan (*missing struktural*) | 1.5c | **Tidak diimputasi**; tahun IPS pertama dipakai memperkirakan angkatan |
| 8 | Beberapa IPS/IPK sangat rendah | 1.9 | **Dipertahankan** (nilai valid yang ekstrem) |
| 9 | Tidak ada duplikat responden | 1.6 | Tidak ada aksi; hasil pengecekan dicatat |
| 10 | 15 responden menjawab 88 pernyataan dengan angka yang sama, 8 lainnya hampir sama | 1.6 | **Diberi tanda (flag)**, tidak dihapus |
| 11 | Belum ada skor maupun label gaya belajar | 1.10 | Dibuat di Bagian 3 (scoring sesuai teori) |

---
# Bagian 2 – Data Cleaning

Bagian ini menangani masalah yang sudah dideteksi di Bagian 1 (lihat tabel 1.11).
Setiap langkah cleaning:
1. dikerjakan lewat kode,
2. dicatat ke `cleaning_log`: langkah, alasan, jumlah baris sebelum/sesudah, dan data yang terdampak.

Data asli (`raw`) tidak diubah. Semua perubahan dilakukan pada salinannya (`df`).

In [19]:
cleaning_log = []

def catat(langkah, aksi, alasan, n_sebelum, n_sesudah, terdampak=""):
    """Mencatat satu langkah cleaning ke dalam cleaning_log."""
    cleaning_log.append({
        "Langkah": langkah,
        "Aksi": aksi,
        "Alasan": alasan,
        "Baris sebelum": n_sebelum,
        "Baris sesudah": n_sesudah,
        "Terdampak": terdampak,
    })

df = raw.copy()   # semua cleaning dikerjakan pada salinan; `raw` tetap asli

print("Salinan data dibuat:", df.shape[0], "baris x", df.shape[1], "kolom")

Salinan data dibuat: 194 baris x 116 kolom


## C1 – Ganti nama kolom
**Masalah (temuan 1.3):** header kolom berupa teks pertanyaan yang panjang, sehingga sulit dipakai di kode.

**Penanganan:** ganti ke kode pendek yang sudah disusun di Bagian 1.2. Teks aslinya disimpan di tabel `kolom_map` supaya setiap kolom tetap bisa dilacak ke pertanyaan aslinya (Spesifikasi 7.3).

| Pola nama | Contoh | Arti |
|---|---|---|
| `VARK_Qxx` | `VARK_Q01` | Pertanyaan VARK ke-1 |
| `<FRAMEWORK>_<TIPE>_xx` | `FS_SEN_03` | Felder-Silverman, Sensing, pernyataan ke-3 |
| `IPS_<tahun>_S<n>` | `IPS_2025_S2` | IPS tahun 2025 semester 2 |

In [20]:
# 1) Simpan pasangan nama baru <-> teks asli (dipakai nanti di data dictionary)
kolom_map = pd.DataFrame({
    "nama_kolom": [nama_baru[c] for c in cols],
    "teks_asli": cols,
    "framework": [fw for nama, fw, a, b, kode in GROUPS for _ in range(a, b + 1)],
})

# 2) Ganti nama kolom di salinan data
df = df.rename(columns=nama_baru)
assert df.columns.is_unique, "Ada nama kolom baru yang kembar!"

# 3) Perbarui daftar kolom per kelompok supaya memakai nama baru
VARK_COLS   = [c for c in df.columns if c.startswith("VARK_Q")]
LIKERT_COLS = [c for c in df.columns if c.startswith(("HM_", "FS_", "KOLB_"))]
IPS_COLS    = [c for c in df.columns if c.startswith("IPS_")]
SURVEY_COLS = VARK_COLS + LIKERT_COLS

# 4) Catat ke cleaning log
catat("C1", "Ganti nama 116 kolom ke kode pendek",
      "Header asli terlalu panjang; teks asli disimpan di kolom_map",
      len(df), len(df))

print("Jumlah kolom:", len(df.columns))
print(f"VARK: {len(VARK_COLS)} | Likert: {len(LIKERT_COLS)} | IPS: {len(IPS_COLS)}")
print("Kolom IPS:", IPS_COLS)
display(kolom_map.iloc[[0, 1, 17, 49, 81, 105, 115]])

Jumlah kolom: 116
VARK: 16 | Likert: 88 | IPS: 10
Kolom IPS: ['IPS_2022_S1', 'IPS_2022_S2', 'IPS_2023_S1', 'IPS_2023_S2', 'IPS_2024_S1', 'IPS_2024_S2', 'IPS_2025_S1', 'IPS_2025_S2', 'IPS_2025_S3', 'IPS_2026_S1']


,nama_kolom,teks_asli,framework
0,ID,ID,Identifier
1,VARK_Q01,"Ketika mempelajari topik baru yang cukup kompleks, saya paling terbantu jika...",VARK
17,HM_ACT_01,Saya menikmati aktivitas belajar baru meskipun belum sepenuhnya tahu bagaima...,Honey-Mumford
49,FS_ACT_01,Saya memahami materi lebih cepat ketika dapat mendiskusikannya atau mengerja...,Felder-Silverman
81,KOLB_CE_01,Saya lebih mudah mulai belajar ketika memiliki pengalaman atau contoh konkre...,Kolb
105,IPS_2022_S1,IPS 2022 Semester 1,Nilai akademik
115,IPK,IPK,Nilai akademik


## C2 – Buang respons kosong total
**Masalah (temuan 1.5a dan 1.6):** ID 11 dan 133 tidak berisi jawaban survei sama sekali, dan IPS/IPK-nya juga kosong. Keduanya juga terdeteksi sebagai "duplikat persis" satu sama lain, padahal itu hanya karena sama-sama kosong.

**Penanganan:** kedua baris dibuang. Tidak ada data yang bisa dianalisis dari baris ini, dan Spesifikasi 7.2 menyebut "respons kosong" sebagai hal yang perlu ditangani. Jumlah sebelum/sesudah dan ID-nya dicatat di cleaning log.

In [21]:
n_sebelum = len(df)

# 1) Cari baris yang SELURUH 104 jawaban surveinya kosong
kosong_total = df[SURVEY_COLS].isna().all(axis=1)
id_kosong = df.loc[kosong_total, "ID"].tolist()

# 2) Tampilkan dulu baris yang akan dibuang (sebagai bukti)
print("Baris yang akan dibuang:")
display(df.loc[kosong_total, ["ID"] + IPS_COLS + ["IPK"]])

# 3) Buang, lalu rapikan nomor index
df = df.loc[~kosong_total].reset_index(drop=True)

# 4) Catat ke cleaning log
catat("C2", "Hapus baris yang seluruh jawaban surveinya kosong",
      "Respons kosong total (survei, IPS, dan IPK kosong); tidak ada data untuk dianalisis",
      n_sebelum, len(df), f"ID {id_kosong}")

print(f"\nJumlah baris: {n_sebelum} -> {len(df)} (dibuang: ID {id_kosong})")
print("Sisa sel kosong di kolom survei:", int(df[SURVEY_COLS].isna().sum().sum()))

Baris yang akan dibuang:


,ID,IPS_2022_S1,IPS_2022_S2,IPS_2023_S1,IPS_2023_S2,IPS_2024_S1,IPS_2024_S2,IPS_2025_S1,IPS_2025_S2,IPS_2025_S3,IPS_2026_S1,IPK
10,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
132,133,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Jumlah baris: 194 -> 192 (dibuang: ID [11, 133])
Sisa sel kosong di kolom survei: 0


## C3 – Validasi jawaban skala 1–5 & perbaiki tipe data
**Masalah (temuan 1.3 dan 1.8):** 88 pernyataan skala 1–5 tersimpan sebagai angka desimal (`float64`, misalnya `4.0`), padahal jawabannya bilangan bulat. Penyebabnya, pandas otomatis memakai desimal untuk kolom angka yang punya sel kosong.

**Penanganan:**
1. Validasi dulu bahwa semua jawaban adalah 1, 2, 3, 4, atau 5.
2. Ubah tipe data menjadi bilangan bulat (`Int64`).

Ini **hanya konversi format penyimpanan**. Nilai jawaban tidak berubah (`4.0` → `4`), sehingga makna jawaban asli tetap terjaga (Spesifikasi 7.3).

In [22]:
# 1) Validasi: semua jawaban harus salah satu dari 1, 2, 3, 4, 5
tidak_valid = ~df[LIKERT_COLS].isin([1, 2, 3, 4, 5])
n_tidak_valid = int(tidak_valid.sum().sum())
print("Jumlah jawaban di luar 1-5:", n_tidak_valid)
assert n_tidak_valid == 0, "Ada jawaban di luar 1-5, periksa dulu sebelum lanjut!"

# 2) Simpan salinan nilai sebelum konversi (untuk bukti bahwa nilainya tidak berubah)
sebelum = df[LIKERT_COLS].copy()

# 3) Ubah tipe data: desimal (float64) -> bilangan bulat (Int64)
print("Tipe data sebelum:", df[LIKERT_COLS].dtypes.unique())
df[LIKERT_COLS] = df[LIKERT_COLS].astype("Int64")
print("Tipe data sesudah:", df[LIKERT_COLS].dtypes.unique())

# 4) Bukti: nilai jawaban sama persis sebelum dan sesudah konversi
print("Nilai jawaban berubah?", not (sebelum == df[LIKERT_COLS].astype(float)).all().all())
display(pd.concat([sebelum.iloc[:3, :4].add_suffix(" (sebelum)"),
                   df[LIKERT_COLS].iloc[:3, :4].add_suffix(" (sesudah)")], axis=1))

# 5) Catat ke cleaning log
catat("C3", "Validasi 88 item Likert & ubah tipe float64 -> Int64",
      f"{n_tidak_valid} nilai di luar 1-5; konversi format saja, nilai jawaban tidak berubah",
      len(df), len(df))

Jumlah jawaban di luar 1-5: 0
Tipe data sebelum: [dtype('float64')]
Tipe data sesudah: [Int64Dtype()]
Nilai jawaban berubah? False


,HM_ACT_01 (sebelum),HM_ACT_02 (sebelum),HM_ACT_03 (sebelum),HM_ACT_04 (sebelum),HM_ACT_01 (sesudah),HM_ACT_02 (sesudah),HM_ACT_03 (sesudah),HM_ACT_04 (sesudah)
0,3.0,4.0,1.0,3.0,3,4,1,3
1,4.0,3.0,3.0,4.0,4,3,3,4
2,5.0,4.0,4.0,3.0,5,4,4,3


## C4 – Validasi & rapikan jawaban VARK
**Masalah (temuan 1.7):** jawaban VARK berupa teks opsi yang digabung dengan tanda `;` dan selalu diakhiri `;` (misalnya `"opsi a;opsi b;"`). Kita juga perlu memastikan tidak ada opsi asing: typo, isian bebas, atau beda huruf besar/kecil.

**Penanganan:**
1. Tetapkan **daftar opsi resmi** setiap pertanyaan beserta modalitasnya (V = Visual, A = Auditori, R = Read/Write, K = Kinestetik). Daftar ini sekaligus menjadi pemetaan yang dipakai saat scoring.
2. Cek setiap opsi yang dipilih responden terhadap daftar resmi.
3. Rapikan format menjadi `"opsi a; opsi b"`. Isi jawaban tidak berubah, hanya pemisahnya yang dirapikan (Spesifikasi 7.2: normalisasi teks secara hati-hati).

In [23]:
# 1) Daftar opsi resmi per pertanyaan dan modalitasnya (V/A/R/K)
VARK_MAP = {
    'VARK_Q01': {
        'melihat diagram, bagan, peta konsep, atau ilustrasi hubungan antarbagian.': 'V',
        'mendengar penjelasan dan dapat berdiskusi atau bertanya secara lisan.': 'A',
        'membaca uraian tertulis, definisi, atau catatan yang terstruktur.': 'R',
        'mencoba contoh, simulasi, praktik, atau melihat penerapan nyata.': 'K',
    },
    'VARK_Q02': {
        'mengikuti gambar langkah demi langkah atau tangkapan layar.': 'V',
        'meminta seseorang menjelaskan langkah-langkahnya secara lisan.': 'A',
        'membaca petunjuk tertulis dan daftar langkah.': 'R',
        'langsung mencoba sambil memperbaiki kesalahan yang muncul.': 'K',
    },
    'VARK_Q03': {
        'membuat peta konsep, skema, warna, atau diagram.': 'V',
        'menjelaskan materi dengan suara keras atau berdiskusi dengan teman.': 'A',
        'menulis ulang ringkasan, membuat daftar, atau membaca catatan.': 'R',
        'berlatih soal, mencoba contoh, atau menghubungkan materi dengan pengalaman.': 'K',
    },
    'VARK_Q04': {
        'ada visualisasi yang menunjukkan hubungan antar konsep.': 'V',
        'penjelasan disampaikan dengan contoh verbal dan tanya jawab.': 'A',
        'tersedia teks atau rumusan tertulis yang dapat saya baca kembali.': 'R',
        'ada demonstrasi, studi kasus, atau aktivitas langsung.': 'K',
    },
    'VARK_Q05': {
        'melihat contoh hasil akhir atau struktur visual tugas.': 'V',
        'mendengarkan penjelasan dosen tentang apa yang harus dilakukan.': 'A',
        'membaca instruksi dan rubrik secara rinci.': 'R',
        'mencoba bagian awal tugas untuk memahami apa yang diminta.': 'K',
    },
    'VARK_Q06': {
        'melihat peta atau petunjuk arah visual.': 'V',
        'bertanya kepada orang lain dan mendengarkan arahannya.': 'A',
        'membaca nama jalan, papan petunjuk, atau instruksi tertulis.': 'R',
        'berjalan mencoba rute sambil menyesuaikan arah.': 'K',
    },
    'VARK_Q07': {
        'grafik, diagram, dan visualisasi pola.': 'V',
        'penjelasan lisan tentang makna hasil.': 'A',
        'tabel, uraian tertulis, dan penjelasan metodologis.': 'R',
        'contoh penerapan data pada kasus nyata.': 'K',
    },
    'VARK_Q08': {
        'menggambar skema atau menunjukkan visual.': 'V',
        'menjelaskannya melalui percakapan.': 'A',
        'menuliskan poin-poin atau memberi bacaan singkat.': 'R',
        'menunjukkan bagaimana ide tersebut bekerja melalui contoh atau demonstrasi.': 'K',
    },
    'VARK_Q09': {
        'infografik, diagram, atau video dengan visual yang jelas.': 'V',
        'rekaman audio, diskusi sinkron, atau penjelasan verbal.': 'A',
        'modul teks, transkrip, dan catatan yang lengkap.': 'R',
        'simulasi, kuis interaktif, latihan, atau aktivitas berbasis kasus.': 'K',
    },
    'VARK_Q10': {
        'membuat sketsa atau memetakan masalah secara visual.': 'V',
        'membicarakan masalah dengan orang lain untuk memperoleh ide.': 'A',
        'menuliskan fakta, alternatif, dan langkah pemecahan.': 'R',
        'mencoba beberapa solusi dan belajar dari hasil percobaan.': 'K',
    },
    'VARK_Q11': {
        'video animasi, diagram interaktif, atau infografik.': 'V',
        'podcast, rekaman kuliah, atau diskusi kelompok.': 'A',
        'artikel, buku, modul, atau catatan tertulis.': 'R',
        'tutorial praktik, proyek, eksperimen, atau studi kasus.': 'K',
    },
    'VARK_Q12': {
        'dapat menggambarkan hubungan konsep tersebut dalam bentuk visual.': 'V',
        'dapat menjelaskannya secara lisan dengan kata-kata saya sendiri.': 'A',
        'dapat menuliskan definisi dan penjelasannya secara runtut.': 'R',
        'dapat menerapkannya untuk menyelesaikan masalah atau tugas nyata.': 'K',
    },
    'VARK_Q13': {
        'mencari bagian visual terlebih dahulu untuk memahami gambaran besar.': 'V',
        'mendengarkan penjelasan terkait sebelum atau sambil belajar.': 'A',
        'membaca dan membuat catatan secara sistematis.': 'R',
        'mencari latihan atau contoh penerapan setelah memahami pokok materi.': 'K',
    },
    'VARK_Q14': {
        'ditunjukkan bagian mana yang perlu diperbaiki secara visual atau dengan markup.': 'V',
        'dibicarakan secara langsung sehingga saya dapat bertanya.': 'A',
        'diberikan komentar tertulis yang rinci.': 'R',
        'diberi kesempatan mencoba revisi dan melihat dampaknya.': 'K',
    },
    'VARK_Q15': {
        'menyusun bagan, desain, atau struktur visual.': 'V',
        'berdiskusi, mempresentasikan, atau mengoordinasikan secara lisan.': 'A',
        'menulis dokumen, rangkuman, atau notulen.': 'R',
        'mengerjakan prototipe, praktik, atau implementasi.': 'K',
    },
    'VARK_Q16': {
        'melihat representasi visual tentang bagaimana hal itu bekerja.': 'V',
        'mendengar seseorang menjelaskannya dan berdialog.': 'A',
        'membaca penjelasan tertulis yang lengkap.': 'R',
        'mengalami atau mencoba secara langsung.': 'K',
    },
}

# Pengaman: setiap pertanyaan harus punya tepat 1 opsi untuk setiap modalitas
for q in VARK_COLS:
    assert sorted(VARK_MAP[q].values()) == ["A", "K", "R", "V"], f"Pemetaan {q} tidak lengkap"

# 2) Validasi: setiap opsi yang dipilih responden harus ada di daftar resmi
opsi_asing = []
for q in VARK_COLS:
    for idx, jawaban in df[q].items():
        for opsi in pecah(jawaban):
            if opsi not in VARK_MAP[q]:
                opsi_asing.append((df.at[idx, "ID"], q, opsi))
print("Jumlah opsi asing (typo / isian bebas / beda kapitalisasi):", len(opsi_asing))
assert not opsi_asing, f"Ada opsi asing, contoh: {opsi_asing[:3]}"

# 3) Rapikan format: buang ';' di akhir dan spasi berlebih -> "opsi a; opsi b"
contoh_sebelum = df.at[0, "VARK_Q01"]
df[VARK_COLS] = df[VARK_COLS].apply(lambda kolom: kolom.apply(lambda s: "; ".join(pecah(s))))
print("\nContoh sebelum:", repr(contoh_sebelum))
print("Contoh sesudah:", repr(df.at[0, "VARK_Q01"]))

# 4) Ringkasan pemetaan opsi -> modalitas untuk 2 pertanyaan pertama
peta = pd.DataFrame([(q, m, o.rstrip(".")) for q in VARK_COLS[:2] for o, m in VARK_MAP[q].items()],
                    columns=["Pertanyaan", "Modalitas", "Opsi"])
display(peta)

# 5) Catat ke cleaning log
catat("C4", "Validasi opsi VARK terhadap daftar resmi & rapikan format",
      f"{len(opsi_asing)} opsi asing; buang ';' di akhir dan spasi berlebih (isi jawaban tidak berubah)",
      len(df), len(df))

Jumlah opsi asing (typo / isian bebas / beda kapitalisasi): 0

Contoh sebelum: 'melihat diagram, bagan, peta konsep, atau ilustrasi hubungan antarbagian.;mendengar penjelasan dan dapat berdiskusi atau bertanya secara lisan.;'
Contoh sesudah: 'melihat diagram, bagan, peta konsep, atau ilustrasi hubungan antarbagian.; mendengar penjelasan dan dapat berdiskusi atau bertanya secara lisan.'


,Pertanyaan,Modalitas,Opsi
0,VARK_Q01,V,"melihat diagram, bagan, peta konsep, atau ilustrasi hubungan antarbagian"
1,VARK_Q01,A,mendengar penjelasan dan dapat berdiskusi atau bertanya secara lisan
2,VARK_Q01,R,"membaca uraian tertulis, definisi, atau catatan yang terstruktur"
3,VARK_Q01,K,"mencoba contoh, simulasi, praktik, atau melihat penerapan nyata"
4,VARK_Q02,V,mengikuti gambar langkah demi langkah atau tangkapan layar
5,VARK_Q02,A,meminta seseorang menjelaskan langkah-langkahnya secara lisan
6,VARK_Q02,R,membaca petunjuk tertulis dan daftar langkah
7,VARK_Q02,K,langsung mencoba sambil memperbaiki kesalahan yang muncul


## C5 – IPS 2026 Semester 1: angka 0 diubah menjadi kosong
**Masalah (temuan 1.5d dan 1.9):** IPS 2026 Semester 1 bernilai 0 untuk semua yang terisi. Semester ini masih berjalan, jadi 0 di sini berarti "nilai belum ada", bukan IPS nol.

**Penanganan:** ubah 0 menjadi kosong (`NaN`). Kalau dibiarkan 0, rata-rata IPS akan anjlok dan seolah-olah ada mahasiswa yang IPS-nya nol.

In [24]:
kolom = "IPS_2026_S1"

# 1) Lihat isi kolom sebelum diubah
print("Isi IPS_2026_S1 sebelum:", df[kolom].value_counts(dropna=False).to_dict())

# 2) Ubah angka 0 menjadi kosong (NaN)
nol = df[kolom] == 0
n_nol = int(nol.sum())
df.loc[nol, kolom] = np.nan

# 3) Cek hasilnya
print("Isi IPS_2026_S1 sesudah:", df[kolom].value_counts(dropna=False).to_dict())

# 4) Catat ke cleaning log
catat("C5", f"IPS_2026_S1: nilai 0 -> NaN ({n_nol} sel)",
      "Semester 2026 S1 masih berjalan; angka 0 berarti nilai belum tersedia, bukan IPS nol",
      len(df), len(df), f"{n_nol} sel")

Isi IPS_2026_S1 sebelum: {0.0: 173, nan: 19}
Isi IPS_2026_S1 sesudah: {nan: 192}


## C6 – IPK = 0 pada mahasiswa baru diubah menjadi kosong
**Masalah (temuan 1.9):** mahasiswa yang belum punya IPS sama sekali tidak mungkin punya IPK. Sebagian dari mereka tercatat IPK = 0, sebagian lagi kosong, padahal kondisinya sama.

**Penanganan:** IPK = 0 pada responden tanpa IPS diubah menjadi kosong, supaya konsisten. IPK yang rendah tetapi punya IPS (misalnya 0,87) **tidak** diubah, karena itu nilai asli.

In [25]:
# Semester yang sudah selesai (tanpa IPS_2026_S1 yang masih berjalan)
IPS_SELESAI = [c for c in IPS_COLS if c != "IPS_2026_S1"]

# 1) Cari mahasiswa yang belum punya IPS sama sekali
tanpa_ips = df[IPS_SELESAI].isna().all(axis=1)
print("Responden tanpa IPS semester yang sudah selesai:", int(tanpa_ips.sum()))
print("  IPK mereka:", df.loc[tanpa_ips, "IPK"].value_counts(dropna=False).to_dict())

# 2) Di antara mereka, yang IPK-nya tercatat 0 -> ubah menjadi kosong
ubah = tanpa_ips & (df["IPK"] == 0)
id_ubah = df.loc[ubah, "ID"].tolist()
df.loc[ubah, "IPK"] = np.nan

# 3) Cek hasilnya
print("\nIPK diubah 0 -> NaN untuk ID:", id_ubah)
print("IPK mereka sekarang:", df.loc[tanpa_ips, "IPK"].value_counts(dropna=False).to_dict())
print("IPK terisi:", int(df["IPK"].notna().sum()), "| IPK kosong:", int(df["IPK"].isna().sum()), "dari", len(df))

# 4) Catat ke cleaning log
catat("C6", "IPK = 0 pada responden tanpa IPS -> NaN",
      "Belum punya IPS berarti belum punya IPK; disamakan dengan kasus serupa yang sudah kosong",
      len(df), len(df), f"ID {id_ubah}")

Responden tanpa IPS semester yang sudah selesai: 19
  IPK mereka: {nan: 15, 0.0: 4}

IPK diubah 0 -> NaN untuk ID: [20, 29, 59, 82]
IPK mereka sekarang: {nan: 19}
IPK terisi: 173 | IPK kosong: 19 dari 192


## C7 – Validasi rentang & konsistensi IPS/IPK
**Yang dicek:**
1. Semua IPS/IPK harus berada di rentang 0–4. Nilai di luar itu tidak mungkin.
2. IPK seharusnya mirip dengan rata-rata IPS. Selisih kecil itu wajar karena IPK dihitung dengan bobot SKS.
3. Nilai yang sangat rendah (IPS < 2) **tidak dihapus**. Nilai ini mungkin terjadi dan valid, hanya ekstrem (Spesifikasi 7.2: bedakan nilai ekstrem dari nilai yang tidak mungkin).

In [26]:
IPS_IPK = IPS_COLS + ["IPK"]

# 1) Validasi rentang: semua IPS/IPK harus 0-4 (yang kosong diabaikan)
di_luar = ((df[IPS_IPK] < 0) | (df[IPS_IPK] > 4)).sum().sum()
print("Nilai IPS/IPK di luar rentang 0-4:", int(di_luar))
assert di_luar == 0, "Ada nilai IPS/IPK yang tidak mungkin!"

# 2) Konsistensi: IPK seharusnya mirip dengan rata-rata IPS
rata_ips = df[IPS_SELESAI].mean(axis=1)
selisih = (df["IPK"] - rata_ips).dropna()
print(f"Selisih IPK - rata-rata IPS: rata-rata {selisih.mean():.3f}, terbesar {selisih.abs().max():.2f}")

# 3) Nilai ekstrem: IPS di bawah 2 -> ditampilkan, TIDAK dihapus
rendah = (df[IPS_SELESAI] < 2).any(axis=1)
print("\nResponden dengan IPS < 2 (dipertahankan):")
display(df.loc[rendah, ["ID"] + IPS_SELESAI + ["IPK"]].round(2))

# 4) Catat ke cleaning log
catat("C7", "Validasi rentang IPS/IPK (0-4) & konsistensi IPK vs rata-rata IPS",
      f"{int(di_luar)} nilai di luar rentang; selisih IPK-rata IPS maks {selisih.abs().max():.2f}; "
      "IPS rendah = nilai valid yang ekstrem, dipertahankan",
      len(df), len(df), f"IPS<2: ID {df.loc[rendah, 'ID'].tolist()}")

Nilai IPS/IPK di luar rentang 0-4: 0
Selisih IPK - rata-rata IPS: rata-rata 0.009, terbesar 0.21

Responden dengan IPS < 2 (dipertahankan):


,ID,IPS_2022_S1,IPS_2022_S2,IPS_2023_S1,IPS_2023_S2,IPS_2024_S1,IPS_2024_S2,IPS_2025_S1,IPS_2025_S2,IPS_2025_S3,IPK
3,4,NaN,NaN,NaN,NaN,3.03,0.95,0.90,2.79,NaN,1.97
6,7,NaN,NaN,NaN,NaN,NaN,NaN,1.95,2.39,NaN,2.17
28,30,NaN,NaN,NaN,NaN,NaN,NaN,0.87,NaN,NaN,0.87
110,112,NaN,NaN,NaN,NaN,2.97,2.74,1.63,2.42,NaN,2.43
126,128,NaN,NaN,NaN,NaN,3.07,3.38,1.96,2.98,NaN,2.95
131,134,NaN,NaN,NaN,NaN,3.31,2.88,2.03,0.94,3.0,2.24
172,175,NaN,NaN,NaN,NaN,2.76,1.28,2.86,2.03,3.0,2.38


## C8 – Cek ulang duplikat
**Temuan 1.6:** tidak ada duplikat responden. Satu-satunya "duplikat persis" adalah ID 11 dan 133, yang sudah dibuang di C2.

**Penanganan:** cek ulang setelah C2 untuk memastikan memang sudah tidak ada duplikat. Tidak ada baris yang dihapus di langkah ini; hasilnya hanya dicatat.

In [27]:
# 1) Cek ulang duplikat setelah 2 baris kosong dibuang
id_ganda   = int(df["ID"].duplicated().sum())
dup_persis = int(df.drop(columns="ID").duplicated().sum())
dup_survei = int(df[SURVEY_COLS].duplicated().sum())
print("ID ganda               :", id_ganda)
print("Duplikat persis        :", dup_persis)
print("Jawaban survei identik :", dup_survei)

# 2) Pasangan "sangat mirip" dari Bagian 1.6: apakah ada yang IPS/IPK-nya sama?
print(f"Pasangan sangat mirip (1.6): {len(tabel_mirip)}, "
      f"yang IPS/IPK-nya sama: {int(tabel_mirip['IPS/IPK sama?'].sum())}")

# 3) Catat ke cleaning log (tidak ada yang dihapus)
catat("C8", "Cek duplikat (ID, persis, survei identik, sangat mirip)",
      f"{id_ganda} ID ganda, {dup_persis} duplikat persis, {dup_survei} survei identik; "
      "pasangan mirip punya IPS/IPK berbeda -> orang berbeda, tidak ada yang dihapus",
      len(df), len(df))

ID ganda               : 0
Duplikat persis        : 0
Jawaban survei identik : 0
Pasangan sangat mirip (1.6): 78, yang IPS/IPK-nya sama: 0


## C9 – Tandai pola jawaban seragam (flag, tidak dihapus)
**Temuan 1.6:** 15 responden menjawab angka yang sama untuk ke-88 pernyataan (*straightlining*), dan 8 lainnya hampir sama (84–87 dari 88). Artinya mereka setuju penuh pada pernyataan yang saling bertolak belakang, misalnya *"suka langsung mencoba"* sekaligus *"suka mengamati dulu"*.

**Penanganan:** mereka **tidak dihapus** (Spesifikasi 7.2: tidak boleh menghapus otomatis), tetapi diberi kolom penanda:

| Kolom | Arti |
|---|---|
| `flag_straightline` | 88 jawaban skala identik |
| `flag_near_straight` | 84–87 dari 88 jawaban identik |
| `row_sd` | Simpangan baku jawaban satu orang (0 = semua sama) |
| `longstring_max` | Deretan jawaban sama berurutan yang paling panjang |

Bagaimana flag ini dipakai diputuskan di tahap scoring dan EDA.

In [28]:
def longstring(jawaban):
    """Panjang deretan jawaban sama yang berurutan paling panjang."""
    terpanjang = sekarang = 1
    for a, b in zip(jawaban[:-1], jawaban[1:]):
        sekarang = sekarang + 1 if a == b else 1
        terpanjang = max(terpanjang, sekarang)
    return terpanjang

L = df[LIKERT_COLS]

# 1) Indikator pola jawaban per responden
df["row_sd"]         = L.astype(float).std(axis=1).round(3)          # 0 = semua jawaban sama
df["longstring_max"] = L.apply(lambda r: longstring(r.to_numpy()), axis=1)
jawaban_terbanyak    = L.apply(lambda r: r.value_counts().iloc[0], axis=1)

# 2) Flag (tanda), bukan penghapusan
df["flag_straightline"]  = L.nunique(axis=1).eq(1)                    # 88 dari 88 sama
df["flag_near_straight"] = ~df["flag_straightline"] & (jawaban_terbanyak >= 84)   # 84-87 sama

sl   = df.loc[df["flag_straightline"]]
near = df.loc[df["flag_near_straight"]]
print("flag_straightline :", len(sl), "orang ->", sl["ID"].tolist())
print("   angka yang dipilih:", {int(k): int(v) for k, v in sl[LIKERT_COLS[0]].value_counts().sort_index().items()})
print("flag_near_straight:", len(near), "orang ->", near["ID"].tolist())

# 3) Bandingkan indikator: straightliner vs hampir vs lainnya
kelompok = np.select([df["flag_straightline"], df["flag_near_straight"]],
                     ["straightline", "hampir straightline"], "lainnya")
display(df.groupby(kelompok)[["row_sd", "longstring_max"]].agg(["min", "median", "max"]).round(2))

# 4) Catat ke cleaning log
catat("C9", "Tambah kolom flag kualitas jawaban (tidak menghapus)",
      "Straightlining = jawaban identik pada pernyataan yang saling berlawanan; dipertahankan dengan flag",
      len(df), len(df), f"flag_straightline: {len(sl)} | flag_near_straight: {len(near)}")

flag_straightline : 15 orang -> [9, 38, 55, 57, 64, 68, 115, 125, 132, 136, 159, 167, 171, 174, 194]
   angka yang dipilih: {3: 5, 4: 5, 5: 5}
flag_near_straight: 8 orang -> [42, 44, 78, 126, 142, 166, 187, 188]


row_sd              longstring_max           
                       min median   max            min median max
hampir straightline   0.11   0.15  0.79             40   65.0  86
lainnya               0.28   0.72  1.65              2    8.0  56
straightline          0.00   0.00  0.00             88   88.0  88

## C10 – Cleaning log & ringkasan
Rekap semua langkah cleaning. Tabel ini masuk ke laporan dan disimpan sebagai file CSV di folder `report/`. Di bawahnya ada perbandingan kondisi data sebelum dan sesudah cleaning.

In [29]:
# 1) Tampilkan cleaning log
log_df = pd.DataFrame(cleaning_log)
with pd.option_context("display.max_colwidth", 120):
    display(log_df)

# 2) Simpan ke folder report/
REPORT_DIR.mkdir(exist_ok=True)
log_df.to_csv(REPORT_DIR / f"{PREFIX}_cleaning_log.csv", index=False)
print(f"Tersimpan: report/{PREFIX}_cleaning_log.csv")

# 3) Ringkasan sebelum vs sesudah cleaning
ringkasan_cleaning = pd.DataFrame({
    "Sebelum (raw)": [raw.shape[0], raw.shape[1], int(raw[cols[1:105]].isna().sum().sum()),
                      int((raw[cols[114]] == 0).sum()), int((raw[cols[115]] == 0).sum()), "-", "-"],
    "Sesudah (df)":  [df.shape[0], df.shape[1], int(df[SURVEY_COLS].isna().sum().sum()),
                      int((df["IPS_2026_S1"] == 0).sum()), int((df["IPK"] == 0).sum()),
                      int(df["flag_straightline"].sum()), int(df["flag_near_straight"].sum())],
}, index=["Jumlah baris", "Jumlah kolom", "Sel kosong di kolom survei",
          "IPS_2026_S1 bernilai 0", "IPK bernilai 0",
          "Diberi flag straightline", "Diberi flag hampir straightline"])
display(ringkasan_cleaning)

,Langkah,Aksi,Alasan,Baris sebelum,Baris sesudah,Terdampak
0,C1,Ganti nama 116 kolom ke kode pendek,Header asli terlalu panjang; teks asli disimpan di kolom_map,194,194,
1,C2,Hapus baris yang seluruh jawaban surveinya kosong,"Respons kosong total (survei, IPS, dan IPK kosong); tidak ada data untuk dianalisis",194,192,"ID [11, 133]"
2,C3,Validasi 88 item Likert & ubah tipe float64 -> Int64,"0 nilai di luar 1-5; konversi format saja, nilai jawaban tidak berubah",192,192,
3,C4,Validasi opsi VARK terhadap daftar resmi & rapikan format,0 opsi asing; buang ';' di akhir dan spasi berlebih (isi jawaban tidak berubah),192,192,
4,C5,IPS_2026_S1: nilai 0 -> NaN (173 sel),"Semester 2026 S1 masih berjalan; angka 0 berarti nilai belum tersedia, bukan IPS nol",192,192,173 sel
5,C6,IPK = 0 pada responden tanpa IPS -> NaN,Belum punya IPS berarti belum punya IPK; disamakan dengan kasus serupa yang sudah kosong,192,192,"ID [20, 29, 59, 82]"
6,C7,Validasi rentang IPS/IPK (0-4) & konsistensi IPK vs rata-rata IPS,"0 nilai di luar rentang; selisih IPK-rata IPS maks 0.21; IPS rendah = nilai valid yang ekstrem, dipertahankan",192,192,"IPS<2: ID [4, 7, 30, 112, 128, 134, 175]"
7,C8,"Cek duplikat (ID, persis, survei identik, sangat mirip)","0 ID ganda, 0 duplikat persis, 0 survei identik; pasangan mirip punya IPS/IPK berbeda -> orang berbeda, tidak ada ya...",192,192,
8,C9,Tambah kolom flag kualitas jawaban (tidak menghapus),Straightlining = jawaban identik pada pernyataan yang saling berlawanan; dipertahankan dengan flag,192,192,flag_straightline: 15 | flag_near_straight: 8


Tersimpan: report/KelpAdam_cleaning_log.csv


,Sebelum (raw),Sesudah (df)
Jumlah baris,194,192
Jumlah kolom,116,120
Sel kosong di kolom survei,208,0
IPS_2026_S1 bernilai 0,173,0
IPK bernilai 0,4,0
Diberi flag straightline,-,15
Diberi flag hampir straightline,-,8


**Kesimpulan Bagian 2:** data sudah bersih. Tersisa **192 responden** dengan jawaban survei lengkap dan format yang konsisten. Nilai 0 yang sebenarnya berarti "belum ada nilai" sudah diubah menjadi kosong. Responden dengan pola jawaban seragam sudah ditandai tanpa dihapus. Tidak ada duplikat.

---
# Bagian 3 – Scoring
## S1 – VARK (Fleming)
**Skor:** jumlah opsi V/A/R/K yang dipilih di 16 soal (0–16).
**Label:** aturan *stepping distance* Fleming. Modalitas yang selisih skornya dengan skor tertinggi ≤ jarak (1–4, tergantung total pilihan) ikut menjadi preferensi.
Format survei sama dengan VARK asli, jadi aturan dipakai apa adanya.

In [30]:
# 1) Skor: hitung berapa kali tiap modalitas dipilih di 16 soal (0-16)
for m in "VARK":
    df[f"VARK_{m}"] = 0
for q in VARK_COLS:
    for idx, jawaban in df[q].items():
        for opsi in pecah(jawaban):
            df.at[idx, f"VARK_{VARK_MAP[q][opsi]}"] += 1
df["VARK_total"] = df[["VARK_V", "VARK_A", "VARK_R", "VARK_K"]].sum(axis=1)

# 2) Stepping distance (Fleming): makin banyak opsi dipilih, makin longgar jaraknya
def stepping_distance(total):
    if total <= 21: return 1
    if total <= 27: return 2
    if total <= 32: return 3
    return 4

# 3) Profil preferensi: ambil yang tertinggi, lalu yang berikutnya selama selisihnya <= stepping distance
def vark_profile(row):
    skor = sorted([(row[f"VARK_{m}"], m) for m in "VARK"], reverse=True)   # dari terbesar
    jarak = stepping_distance(row["VARK_total"])
    pref = [skor[0][1]]
    for i in range(3):
        if skor[i][0] - skor[i + 1][0] > jarak:
            break
        pref.append(skor[i + 1][1])
    return "".join(m for m in "VARK" if m in pref)                         # urutan baku V-A-R-K

df["VARK_profile"] = df.apply(vark_profile, axis=1)
df["VARK_type"] = df["VARK_profile"].str.len().map({1: "Single", 2: "Bimodal", 3: "Trimodal", 4: "Quadmodal"})

# 4) Validasi & contoh
assert df["VARK_total"].between(16, 64).all()
r = df.loc[df["ID"] == 4].iloc[0]
print(f"Contoh ID 4: V={r.VARK_V} A={r.VARK_A} R={r.VARK_R} K={r.VARK_K} | total={r.VARK_total} "
      f"-> stepping={stepping_distance(r.VARK_total)} -> profil={r.VARK_profile} ({r.VARK_type})")
display(df[["VARK_V", "VARK_A", "VARK_R", "VARK_K", "VARK_total"]].describe().loc[["mean", "min", "max"]].round(2))
print("VARK_type:", df["VARK_type"].value_counts().to_dict())
print("Single   :", df.loc[df["VARK_type"] == "Single", "VARK_profile"].value_counts().to_dict())

Contoh ID 4: V=4 A=3 R=4 K=15 | total=26 -> stepping=2 -> profil=K (Single)


,VARK_V,VARK_A,VARK_R,VARK_K,VARK_total
mean,7.66,8.14,7.28,8.07,31.14
min,0.00,0.00,0.00,0.00,16.00
max,16.00,16.00,16.00,16.00,64.00


VARK_type: {'Quadmodal': 74, 'Single': 57, 'Bimodal': 31, 'Trimodal': 30}
Single   : {'K': 21, 'V': 16, 'A': 16, 'R': 4}


## S2 – Felder-Silverman (ILS)
**Skor:** jumlah 4 item kutub a − jumlah 4 item kutub b (−16…+16), sama dengan rumus ILS (a − b).
**Arah (label utama):** + → kutub a, − → kutub b, 0 → NEUTRAL.
**Kekuatan:** ambang ILS 1–3 / 5–7 / 9–11 (dari maks 11) diskalakan ke maks 16 → Balanced 0–5, Moderate 6–11, Strong 12–16.
Straightliner: skor dihitung, label dikosongkan.

In [31]:
# Responden yang labelnya bisa ditentukan (bukan straightliner)
ok = ~df["flag_straightline"]

# Pasangan dimensi: (kutub a, kutub b)
FS_DIMS = {
    "FS_ACT_REF": ("ACT", "REF"),
    "FS_SEN_INT": ("SEN", "INT"),
    "FS_VIS_VRB": ("VIS", "VRB"),
    "FS_SEQ_GLO": ("SEQ", "GLO"),
}

for dim, (a, b) in FS_DIMS.items():
    item_a = [c for c in LIKERT_COLS if c.startswith(f"FS_{a}_")]
    item_b = [c for c in LIKERT_COLS if c.startswith(f"FS_{b}_")]
    # 1) Skor = jumlah item kutub a - jumlah item kutub b  (rentang -16..+16)
    skor = df[item_a].sum(axis=1) - df[item_b].sum(axis=1)
    df[f"{dim}_score"] = skor.astype(int)
    # 2) Arah: + -> kutub a, - -> kutub b, 0 -> NEUTRAL
    df[f"{dim}_pole"] = np.select([skor > 0, skor < 0], [a, b], "NEUTRAL")
    # 3) Kekuatan: ambang ILS (1-3 / 5-7 / 9-11 dari maks 11) diskalakan ke maks 16
    df[f"{dim}_strength"] = pd.cut(skor.abs(), bins=[-1, 5, 11, 16],
                                   labels=["Balanced", "Moderate", "Strong"]).astype(str)
    # 4) Straightliner: label tidak dapat ditentukan
    df.loc[~ok, [f"{dim}_pole", f"{dim}_strength"]] = np.nan

# Validasi & contoh
assert all(df[f"{d}_score"].between(-16, 16).all() for d in FS_DIMS)
cols_fs = [f"{d}_{x}" for d in FS_DIMS for x in ("score", "pole", "strength")]
print("Contoh ID 4:")
display(df.loc[df["ID"] == 4, ["ID"] + cols_fs].set_index("ID").T)

rekap = pd.DataFrame([
    {"Dimensi": d,
     "Kutub a": f"{a} = {(df.loc[ok, f'{d}_pole'] == a).sum()}",
     "NEUTRAL": int((df.loc[ok, f"{d}_pole"] == "NEUTRAL").sum()),
     "Kutub b": f"{b} = {(df.loc[ok, f'{d}_pole'] == b).sum()}",
     **{s: int((df.loc[ok, f"{d}_strength"] == s).sum()) for s in ["Balanced", "Moderate", "Strong"]}}
    for d, (a, b) in FS_DIMS.items()
]).set_index("Dimensi")
print(f"Distribusi label (n = {int(ok.sum())}, tanpa straightliner):")
display(rekap)

Contoh ID 4:


ID,4
FS_ACT_REF_score,8
FS_ACT_REF_pole,ACT
FS_ACT_REF_strength,Moderate
FS_SEN_INT_score,12
FS_SEN_INT_pole,SEN
FS_SEN_INT_strength,Strong
FS_VIS_VRB_score,1
FS_VIS_VRB_pole,VIS
FS_VIS_VRB_strength,Balanced
FS_SEQ_GLO_score,8


Distribusi label (n = 177, tanpa straightliner):


,Kutub a,NEUTRAL,Kutub b,Balanced,Moderate,Strong
Dimensi,,,,,,
FS_ACT_REF,ACT = 67,46,REF = 64,168,8,1
FS_SEN_INT,SEN = 108,36,INT = 33,156,20,1
FS_VIS_VRB,VIS = 81,51,VRB = 45,157,19,1
FS_SEQ_GLO,SEQ = 86,50,GLO = 41,159,18,0


## S3 – Kolb (LSI)
**Skor:** jumlah 6 item per tahap (CE, RO, AC, AE), lalu dua sumbu: AC−CE (berpikir vs merasakan) dan AE−RO (mencoba vs mengamati).
**Label:** 4 tipe dari kombinasi dua sumbu. Batasnya **median sampel**, mengikuti cara manual LSI 3.1 (cut-point = persentil ke-50 kelompok norma). Nilai yang sama dengan batas masuk sisi konkret/reflektif, sesuai manual.
Straightliner: skor dihitung, label dikosongkan.

In [32]:
# 1) Skor tiap tahap = jumlah 6 item (rentang 6..30)
for mode in ["CE", "RO", "AC", "AE"]:
    item = [c for c in LIKERT_COLS if c.startswith(f"KOLB_{mode}_")]
    df[f"KOLB_{mode}_sum"] = df[item].sum(axis=1).astype(int)

# 2) Dua sumbu Kolb (rentang -24..+24)
df["KOLB_ACCE"] = df["KOLB_AC_sum"] - df["KOLB_CE_sum"]   # + = berpikir (abstrak), - = merasakan (konkret)
df["KOLB_AERO"] = df["KOLB_AE_sum"] - df["KOLB_RO_sum"]   # + = mencoba (aktif),    - = mengamati (reflektif)

# 3) Batas = median sampel (cara LSI 3.1: cut-point = persentil ke-50 kelompok norma)
CUT_ACCE = df.loc[ok, "KOLB_ACCE"].median()
CUT_AERO = df.loc[ok, "KOLB_AERO"].median()
print(f"Batas (median, n={int(ok.sum())}): AC-CE = {CUT_ACCE}, AE-RO = {CUT_AERO}")

# 4) Tipe Kolb (nilai = batas masuk sisi konkret/reflektif, sesuai manual)
abstrak = df["KOLB_ACCE"] > CUT_ACCE
aktif   = df["KOLB_AERO"] > CUT_AERO
df["KOLB_style"] = np.select(
    [~abstrak & ~aktif, ~abstrak & aktif, abstrak & ~aktif, abstrak & aktif],
    ["Diverging", "Accommodating", "Assimilating", "Converging"], default="")
df.loc[~ok, "KOLB_style"] = np.nan                        # straightliner: label dikosongkan

# 5) Validasi & contoh
assert df[["KOLB_ACCE", "KOLB_AERO"]].stack().between(-24, 24).all()
cols_kolb = ["KOLB_CE_sum", "KOLB_RO_sum", "KOLB_AC_sum", "KOLB_AE_sum", "KOLB_ACCE", "KOLB_AERO", "KOLB_style"]
print("Contoh ID 4:")
display(df.loc[df["ID"] == 4, ["ID"] + cols_kolb].set_index("ID"))

print(f"Nilai tepat di batas -> AC-CE: {int((df.loc[ok, 'KOLB_ACCE'] == CUT_ACCE).sum())}, "
      f"AE-RO: {int((df.loc[ok, 'KOLB_AERO'] == CUT_AERO).sum())} responden")
print(f"Distribusi KOLB_style (n = {int(ok.sum())}):", df.loc[ok, "KOLB_style"].value_counts().to_dict())
display(pd.crosstab(
    np.where(abstrak[ok], "Berpikir (AC-CE > 0)", "Merasakan (AC-CE <= 0)"),
    np.where(aktif[ok], "Mencoba (AE-RO > 0)", "Mengamati (AE-RO <= 0)"),
    rownames=["Sumbu AC-CE"], colnames=["Sumbu AE-RO"]))

Batas (median, n=177): AC-CE = 0.0, AE-RO = 0.0
Contoh ID 4:


,KOLB_CE_sum,KOLB_RO_sum,KOLB_AC_sum,KOLB_AE_sum,KOLB_ACCE,KOLB_AERO,KOLB_style
ID,,,,,,,
4,30,18,26,30,-4,12,Accommodating


Nilai tepat di batas -> AC-CE: 42, AE-RO: 35 responden
Distribusi KOLB_style (n = 177): {'Diverging': 79, 'Accommodating': 42, 'Assimilating': 38, 'Converging': 18}


Sumbu AE-RO,Mencoba (AE-RO > 0),Mengamati (AE-RO <= 0)
Sumbu AC-CE,,
Berpikir (AC-CE > 0),18,38
Merasakan (AC-CE <= 0),42,79


## S4 – Honey-Mumford (LSQ)
**Skor:** jumlah 8 item per gaya (8–40).
**Label:** skor dibandingkan dengan responden lain (persentil), lalu dibagi ke pita preferensi LSQ 10/20/40/20/10% (Very low → Very strong). Norma LSQ asli tidak dipakai karena formatnya berbeda (centang 80 item) dan menghasilkan 58% "Very strong Activist".
**Gaya dominan:** gaya dengan persentil tertinggi, yaitu yang paling menonjol relatif terhadap norma (prinsip LSQ).
Straightliner: skor dihitung, label dikosongkan.

In [33]:
HM = {"HM_ACT": "Activist", "HM_REF": "Reflector", "HM_THE": "Theorist", "HM_PRA": "Pragmatist"}
BAND = ["Very low", "Low", "Moderate", "Strong", "Very strong"]

for g in HM:
    item = [c for c in LIKERT_COLS if c.startswith(f"{g}_")]
    # 1) Skor gaya = jumlah 8 item (rentang 8..40)
    df[f"{g}_sum"] = df[item].sum(axis=1).astype(int)
    # 2) Posisi relatif (persentil) dibanding responden lain = norma lokal
    df.loc[ok, f"{g}_pct"] = df.loc[ok, f"{g}_sum"].rank(pct=True, method="average").round(3)
    # 3) Pita preferensi LSQ: 10% / 20% / 40% / 20% / 10%
    df.loc[ok, f"{g}_band"] = pd.cut(df.loc[ok, f"{g}_pct"], bins=[0, .10, .30, .70, .90, 1.0001],
                                     labels=BAND).astype(str)

# 4) Gaya dominan = persentil tertinggi (paling menonjol relatif terhadap norma); seri -> digabung
pct_cols = [f"{g}_pct" for g in HM]
def dominan(row):
    if row[pct_cols].isna().any():
        return np.nan                                   # straightliner
    top = row[pct_cols].max()
    return "+".join(HM[g] for g in HM if row[f"{g}_pct"] == top)
df["HM_dominant"] = df.apply(dominan, axis=1)

# 5) Validasi & contoh
assert all(df[f"{g}_sum"].between(8, 40).all() for g in HM)
print("Contoh ID 4:")
contoh = df.loc[df["ID"] == 4].iloc[0]
display(pd.DataFrame({"Skor (8-40)": [contoh[f"{g}_sum"] for g in HM],
                      "Persentil": [contoh[f"{g}_pct"] for g in HM],
                      "Pita": [contoh[f"{g}_band"] for g in HM]}, index=HM.values()))
print("-> dominan:", contoh["HM_dominant"])

print(f"\nPita preferensi per gaya (n = {int(ok.sum())}):")
display(pd.DataFrame({HM[g]: df.loc[ok, f"{g}_band"].value_counts() for g in HM}).reindex(BAND).fillna(0).astype(int))
print("Distribusi HM_dominant:", df.loc[ok, "HM_dominant"].value_counts().to_dict())

Contoh ID 4:


,Skor (8-40),Persentil,Pita
Activist,40,0.975,Very strong
Reflector,27,0.161,Low
Theorist,36,0.805,Strong
Pragmatist,40,0.972,Very strong


-> dominan: Activist

Pita preferensi per gaya (n = 177):


,Activist,Reflector,Theorist,Pragmatist
Very low,21,17,18,18
Low,33,37,35,29
Moderate,70,67,75,71
Strong,35,36,28,41
Very strong,18,20,21,18


Distribusi HM_dominant: {'Reflector': 51, 'Activist': 48, 'Theorist': 37, 'Pragmatist': 37, 'Activist+Reflector': 2, 'Theorist+Pragmatist': 2}


---
# Bagian 4 – Data Preparation
## P1 – Variabel konteks: perkiraan angkatan
Dataset tidak punya prodi/angkatan, padahal EDA butuh variabel konteks. Angkatan diperkirakan dari **tahun IPS pertama** (temuan 1.5c). Ini **perkiraan (derived)**, bukan data resmi.
Angkatan 2022 hanya 1 orang, jadi digabung ke "<=2023" supaya tidak bisa dikenali.

In [34]:
# 1) Tahun IPS pertama dari semester yang sudah selesai (IPS_SELESAI dari C6)
punya_ips = df[IPS_SELESAI].notna()
tahun_pertama = punya_ips.idxmax(axis=1).str.split("_").str[1].where(punya_ips.any(axis=1))

# 2) Mahasiswa baru: belum punya IPS selesai, tapi tercatat di IPS 2026 S1 pada data asli
tercatat_2026 = df["ID"].map(raw.set_index(cols[0])[cols[114]].notna())
tahun_pertama = tahun_pertama.where(tahun_pertama.notna(), np.where(tercatat_2026, "2026", None))

# 3) Kelompokkan: angkatan 2022 hanya 1 orang -> digabung ke "<=2023" agar tidak bisa dikenali
df["angkatan_inferred"] = (tahun_pertama.replace({"2022": "<=2023", "2023": "<=2023"})
                           .fillna("Tidak diketahui"))
df["n_semester_ips"] = punya_ips.sum(axis=1)        # jumlah semester yang sudah ada IPS-nya

# 4) Cek hasil
urutan = ["<=2023", "2024", "2025", "2026", "Tidak diketahui"]
display(df.groupby("angkatan_inferred").agg(
            Responden=("ID", "size"),
            Semester_IPS=("n_semester_ips", "median"),
            IPK_terisi=("IPK", "count")).reindex(urutan))

,Responden,Semester_IPS,IPK_terisi
angkatan_inferred,,,
<=2023,18,6.0,18
2024,107,4.0,107
2025,48,2.0,48
2026,11,0.0,0
Tidak diketahui,8,0.0,0


## P2 – Dataset prepared
Kolom dikelompokkan per jenis (Spesifikasi 7.3): identifier, metadata non-identifying, raw survey item, score, label/profile, dan prepared variable. Hasilnya disimpan ke `dataset/KelpAdam_survey_prepared.csv`, **terpisah** dari file asli di `dataset/raw/`.
Item mentah tetap disertakan untuk audit, dan maknanya tidak diubah.

In [35]:
FS_D = ["FS_ACT_REF", "FS_SEN_INT", "FS_VIS_VRB", "FS_SEQ_GLO"]
HM_G = ["HM_ACT", "HM_REF", "HM_THE", "HM_PRA"]
KM   = ["CE", "RO", "AC", "AE"]

# 1) Kelompokkan kolom berdasarkan jenis (5 jenis di Spesifikasi 7.3, ditambah ID & flag kualitas)
JENIS_KOLOM = {
    "Identifier":               ["ID"],
    "Metadata non-identifying": IPS_COLS + ["IPK"],
    "Raw survey item":          VARK_COLS + LIKERT_COLS,
    "Score":                    ["VARK_V", "VARK_A", "VARK_R", "VARK_K", "VARK_total"]
                                + [f"{d}_score" for d in FS_D]
                                + [f"KOLB_{m}_sum" for m in KM] + ["KOLB_ACCE", "KOLB_AERO"]
                                + [f"{g}_{x}" for g in HM_G for x in ("sum", "pct")],
    "Label/profile":            ["VARK_profile", "VARK_type"]
                                + [f"{d}_{x}" for d in FS_D for x in ("pole", "strength")]
                                + ["KOLB_style"] + [f"{g}_band" for g in HM_G] + ["HM_dominant"],
    "Prepared variable":        ["angkatan_inferred", "n_semester_ips",
                                 "row_sd", "longstring_max", "flag_straightline", "flag_near_straight"],
}

# 2) Pastikan setiap kolom df masuk tepat satu jenis
semua = [c for kol in JENIS_KOLOM.values() for c in kol]
assert len(semua) == len(set(semua)), "Ada kolom yang masuk dua jenis"
assert set(semua) == set(df.columns), f"Kolom belum dikelompokkan: {set(df.columns) - set(semua)}"

# 3) Susun dataset prepared dengan urutan kolom per jenis
prepared = df[semua].copy()
display(pd.DataFrame({"Jumlah kolom": {j: len(k) for j, k in JENIS_KOLOM.items()}}))

# 4) Simpan ke dataset/ (terpisah dari file asli di dataset/raw/)
PREPARED_FILE = PROJECT_DIR / "dataset" / f"{PREFIX}_survey_prepared.csv"
prepared.to_csv(PREPARED_FILE, index=False)

# 5) Cek: baca ulang file yang tersimpan
cek = pd.read_csv(PREPARED_FILE)
print(f"Tersimpan: dataset/{PREPARED_FILE.name} -> {cek.shape[0]} baris x {cek.shape[1]} kolom")
print("File asli tetap di:", RAW_FILE.relative_to(PROJECT_DIR))

,Jumlah kolom
Identifier,1
Metadata non-identifying,11
Raw survey item,104
Score,23
Label/profile,16
Prepared variable,6


Tersimpan: dataset/KelpAdam_survey_prepared.csv -> 192 baris x 161 kolom
File asli tetap di: dataset\raw\Survei_Gaya_Belajar_IPS_IPK_NO_PII.xlsx


## P3 – Data dictionary
Setiap kolom di dataset prepared dijelaskan: jenis, tipe data, arti, nilai valid, nilai yang benar-benar muncul, raw/derived, dan sumber aturannya (Spesifikasi 7.3). Dictionary dibuat otomatis dari kode, lalu disimpan ke `dictionary/KelpAdam_data_dictionary.xlsx`.

In [36]:
TEKS_ASLI = dict(zip(kolom_map["nama_kolom"], kolom_map["teks_asli"]))
NAMA = {"ACT": "Active", "REF": "Reflective", "SEN": "Sensing", "INT": "Intuitive", "VIS": "Visual",
        "VRB": "Verbal", "SEQ": "Sequential", "GLO": "Global", "CE": "Concrete Experience",
        "RO": "Reflective Observation", "AC": "Abstract Conceptualization", "AE": "Active Experimentation",
        "THE": "Theorist", "PRA": "Pragmatist", "V": "Visual", "A": "Aural", "R": "Read/Write", "K": "Kinesthetic"}
HM_NAMA = {"HM_ACT": "Activist", "HM_REF": "Reflector", "HM_THE": "Theorist", "HM_PRA": "Pragmatist"}
SUMBER = {"VARK": "Fleming (VARK v7.8): hitungan pilihan + stepping distance",
          "FS": "Felder & Soloman (ILS): skor = a - b; ambang 1-3/5-7/9-11 diskalakan ke maks 16",
          "KOLB": "Kolb & Kolb (2005) LSI 3.1: AC-CE, AE-RO; cut-point = median sampel",
          "HM": "Honey & Mumford (LSQ): skor per gaya; pita persentil 10/20/40/20/10%"}

def jelaskan(c):
    """Kembalikan (arti, nilai_valid, raw/derived, sumber_aturan) untuk satu kolom."""
    if c == "ID":
        return "Nomor urut responden (bukan identitas pribadi)", "1-194, unik", "Raw", "Dataset dosen"
    if c.startswith("IPS_"):
        _, th, sm = c.split("_")
        cat = {"IPS_2026_S1": "Semester berjalan: 0 -> NaN (C5), kolom kosong seluruhnya",
               "IPS_2025_S3": "Kemungkinan semester pendek"}.get(c, "Kosong = belum kuliah di semester itu")
        return f"IPS tahun {th} semester {sm[1:]}. {cat}", "0-4 atau kosong", "Raw (dibersihkan)", "Dataset dosen"
    if c == "IPK":
        return "IPK kumulatif. IPK 0 tanpa IPS -> NaN (C6)", "0-4 atau kosong", "Raw (dibersihkan)", "Dataset dosen"
    if c.startswith("VARK_Q"):
        return TEKS_ASLI[c], "1-4 opsi dari 4 opsi resmi (VARK_MAP, C4), dipisah '; '", "Raw (format dirapikan)", "Kuesioner"
    if c in LIKERT_COLS:
        return TEKS_ASLI[c], "1, 2, 3, 4, 5", "Raw (tipe Int64, C3)", "Kuesioner"
    if c in ["VARK_V", "VARK_A", "VARK_R", "VARK_K"]:
        return f"Jumlah opsi {NAMA[c[-1]]} yang dipilih di 16 soal", "0-16", "Derived", SUMBER["VARK"]
    if c == "VARK_total":
        return "Total opsi VARK yang dipilih", "16-64", "Derived", SUMBER["VARK"]
    if c == "VARK_profile":
        return "Modalitas preferensi (urutan V-A-R-K)", "Kombinasi V/A/R/K, mis. K, VA, VARK", "Derived", SUMBER["VARK"]
    if c == "VARK_type":
        return "Jumlah modalitas preferensi", "Single/Bimodal/Trimodal/Quadmodal", "Derived", SUMBER["VARK"]
    if c.startswith("FS_") and c.count("_") == 3:
        _, a, b, x = c.split("_")
        arti = {"score": f"Jumlah item {NAMA[a]} - jumlah item {NAMA[b]}",
                "pole": f"Arah preferensi {NAMA[a]} vs {NAMA[b]}",
                "strength": f"Kekuatan preferensi {NAMA[a]}/{NAMA[b]}"}[x]
        valid = {"score": "-16 s.d. +16", "pole": f"{a}/{b}/NEUTRAL; kosong = straightliner",
                 "strength": "Balanced (0-5)/Moderate (6-11)/Strong (12-16); kosong = straightliner"}[x]
        return arti, valid, "Derived", SUMBER["FS"]
    if c.startswith("KOLB_") and c.endswith("_sum"):
        m = c.split("_")[1]
        return f"Jumlah 6 item {NAMA[m]} ({m})", "6-30", "Derived", SUMBER["KOLB"]
    if c in ["KOLB_ACCE", "KOLB_AERO"]:
        arti = {"KOLB_ACCE": "AC - CE (+ abstrak/berpikir, - konkret/merasakan)",
                "KOLB_AERO": "AE - RO (+ aktif/mencoba, - reflektif/mengamati)"}[c]
        return arti, "-24 s.d. +24", "Derived", SUMBER["KOLB"]
    if c == "KOLB_style":
        return (f"Tipe Kolb (batas AC-CE = {CUT_ACCE:g}, AE-RO = {CUT_AERO:g})",
                "Diverging/Assimilating/Converging/Accommodating; kosong = straightliner", "Derived", SUMBER["KOLB"])
    if c.startswith("HM_") and c.rsplit("_", 1)[0] in HM_NAMA:
        g, x = c.rsplit("_", 1)
        arti = {"sum": f"Jumlah 8 item {HM_NAMA[g]}",
                "pct": f"Persentil skor {HM_NAMA[g]} dibanding responden lain (norma lokal)",
                "band": f"Pita preferensi {HM_NAMA[g]}"}[x]
        valid = {"sum": "8-40", "pct": "0-1; kosong = straightliner",
                 "band": "Very low/Low/Moderate/Strong/Very strong; kosong = straightliner"}[x]
        return arti, valid, "Derived", SUMBER["HM"]
    if c == "HM_dominant":
        return "Gaya Honey-Mumford dengan persentil tertinggi (seri digabung '+')", \
               "Activist/Reflector/Theorist/Pragmatist; kosong = straightliner", "Derived", SUMBER["HM"]
    lain = {
        "angkatan_inferred": ("Perkiraan angkatan dari tahun IPS pertama (P1)", "<=2023/2024/2025/2026/Tidak diketahui",
                              "Derived", "Inferensi kelompok"),
        "n_semester_ips":    ("Jumlah semester yang sudah ada IPS-nya (tanpa 2026 S1)", "0-9", "Derived", "Inferensi kelompok"),
        "row_sd":            ("Simpangan baku 88 jawaban Likert per responden (C9)", ">= 0 (0 = semua sama)", "Derived", "Pemeriksaan kualitas"),
        "longstring_max":    ("Deretan jawaban Likert sama terpanjang (C9)", "1-88", "Derived", "Pemeriksaan kualitas"),
        "flag_straightline": ("88 jawaban Likert identik (C9)", "True/False", "Derived", "Pemeriksaan kualitas"),
        "flag_near_straight":("84-87 dari 88 jawaban Likert identik (C9)", "True/False", "Derived", "Pemeriksaan kualitas"),
    }
    return lain[c]

# 1) Susun data dictionary untuk semua kolom dataset prepared
jenis_of = {c: j for j, kol in JENIS_KOLOM.items() for c in kol}
def teramati(s):
    s = s.dropna()
    if s.empty: return "kosong seluruhnya"
    if pd.api.types.is_bool_dtype(s) or not pd.api.types.is_numeric_dtype(s):
        return f"{s.nunique()} kategori" if s.nunique() > 6 else ", ".join(map(str, sorted(s.unique())))
    return f"{s.min():g} - {s.max():g}"

baris = []
for c in prepared.columns:
    arti, valid, asal, sumber = jelaskan(c)
    baris.append({"nama_kolom": c, "jenis": jenis_of[c], "tipe_data": str(prepared[c].dtype),
                  "arti": arti, "nilai_valid": valid, "nilai_teramati": teramati(prepared[c]),
                  "terisi": int(prepared[c].notna().sum()), "raw_atau_derived": asal, "sumber_aturan": sumber})
data_dictionary = pd.DataFrame(baris)

# 2) Simpan ke dictionary/
DICT_FILE = PROJECT_DIR / "dictionary" / f"{PREFIX}_data_dictionary.xlsx"
DICT_FILE.parent.mkdir(exist_ok=True)
data_dictionary.to_excel(DICT_FILE, index=False, sheet_name="data_dictionary")
print(f"Tersimpan: dictionary/{DICT_FILE.name} -> {len(data_dictionary)} kolom terdokumentasi")

# 3) Tampilkan contoh: 1-2 baris dari setiap jenis
with pd.option_context("display.max_colwidth", 60):
    display(data_dictionary.groupby("jenis", sort=False).head(2)
            [["nama_kolom", "jenis", "tipe_data", "arti", "nilai_valid", "nilai_teramati", "raw_atau_derived"]])

Tersimpan: dictionary/KelpAdam_data_dictionary.xlsx -> 161 kolom terdokumentasi


,nama_kolom,jenis,tipe_data,arti,nilai_valid,nilai_teramati,raw_atau_derived
0,ID,Identifier,int64,Nomor urut responden (bukan identitas pribadi),"1-194, unik",1 - 194,Raw
1,IPS_2022_S1,Metadata non-identifying,float64,IPS tahun 2022 semester 1. Kosong = belum kuliah di seme...,0-4 atau kosong,3.02631 - 3.02631,Raw (dibersihkan)
2,IPS_2022_S2,Metadata non-identifying,float64,IPS tahun 2022 semester 2. Kosong = belum kuliah di seme...,0-4 atau kosong,3.13333 - 3.13333,Raw (dibersihkan)
12,VARK_Q01,Raw survey item,str,"Ketika mempelajari topik baru yang cukup kompleks, saya ...","1-4 opsi dari 4 opsi resmi (VARK_MAP, C4), dipisah '; '",41 kategori,Raw (format dirapikan)
13,VARK_Q02,Raw survey item,str,Saat harus memahami prosedur penggunaan suatu perangkat ...,"1-4 opsi dari 4 opsi resmi (VARK_MAP, C4), dipisah '; '",35 kategori,Raw (format dirapikan)
116,VARK_V,Score,int64,Jumlah opsi Visual yang dipilih di 16 soal,0-16,0 - 16,Derived
117,VARK_A,Score,int64,Jumlah opsi Aural yang dipilih di 16 soal,0-16,0 - 16,Derived
139,VARK_profile,Label/profile,str,Modalitas preferensi (urutan V-A-R-K),"Kombinasi V/A/R/K, mis. K, VA, VARK",15 kategori,Derived
140,VARK_type,Label/profile,str,Jumlah modalitas preferensi,Single/Bimodal/Trimodal/Quadmodal,"Bimodal, Quadmodal, Single, Trimodal",Derived
155,angkatan_inferred,Prepared variable,object,Perkiraan angkatan dari tahun IPS pertama (P1),<=2023/2024/2025/2026/Tidak diketahui,"2024, 2025, 2026, <=2023, Tidak diketahui",Derived


**Kesimpulan Bagian 4:** dataset prepared (192 baris × 161 kolom) dan data dictionary sudah tersimpan. Setiap kolom tercatat jenis, arti, nilai valid, dan sumber aturannya.